## Requisitos del entorno

Ejecutar este notebook en un entorno con `pandas`, `numpy`, `scikit-learn`, `statsmodels`, `seaborn`, `matplotlib`, `plotly`, `openpyxl`, `joblib` y `gdown` previamente instalados.

> **Secuencia de ejecución CRISP-DM:**
> 1. `01_carga_datos.ipynb — Fase 1: Comprensión del negocio + Carga de datos`
> 2. `02_limpieza_variables.ipynb — Fase 2: Comprensión y preparación de datos`
> 3. `03_universo_analitico.ipynb — Fase 3: Preparación de datos (integración y variables económicas)`
> 4. `04_modelado.ipynb — Fase 4: Modelado (WLS + KMeans)`
> 5. `05_modelo_hibrido.ipynb — Fase 4b: Modelo Híbrido (residuos + clasificador)`
> 6. `06_evaluacion_implementacion.ipynb — Fase 5-6: Evaluación + Implementación`

> ⚠️ **Este notebook requiere haber ejecutado `01_carga_datos.ipynb` previamente.**

> ➡️ **Siguiente notebook: `03_universo_analitico.ipynb`**

![Universidad Central](https://universidad.ucentral.edu.co/tulengua/wp-content/themes/tulengua/images/logo-ucentral.png)

<h2 align="center">Procesamiento y análisis de datos</h2>

<table>
<tr>
<td style="width: 75%; vertical-align: middle;">

## Estimación del gasto personal como variable principal para la evaluación de la capacidad de endeudamiento a partir de la caracterización de los hogares desde la analítica de datos.

**FASE 2 — Comprensión y Preparación de Datos: Limpieza, Homologación y Variables Analíticas**

**CRISP-DM Fase 2-3a:** Limpieza, homologación y construcción de variables analíticas. Incluye nota metodológica sobre diseño muestral ENPH y factor de expansión FEX_C.

</td>
<td style="width: 25%; text-align: center;">

<img src="https://raw.githubusercontent.com/lpaolav/bases-de-datos/main/gasto_personal.png" width="150">

</td>
</tr>
</table>

---

### Estudiantes:
- Oscar Leonardo Duarte Urrego  
- Paola Andrea Velandia Lozano  

### Director de Tesis:
- Miguel Hernández Bejarano

---


---
## 📦 Carga de datos — NB02
> Ejecuta esta celda primero si no vienes de ejecutar `01_carga_datos.ipynb` en la misma sesión.

In [1]:

# ============================================================
# CARGA DE DATOS — INICIO NB02
# Requiere haber ejecutado NB01 (01_carga_datos.ipynb)
# ============================================================
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import os, warnings
from pandas.errors import SettingWithCopyWarning

warnings.filterwarnings("ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

SEED = int(os.getenv("TESIS_RANDOM_STATE", "42"))
_cwd = Path.cwd()
BASE_PATH = Path(os.getenv("TESIS_BASE_PATH",
    _cwd.parent if _cwd.name in ("notebooks", "notebook") else _cwd)).resolve()
PERSIST_DIR = BASE_PATH / "02_intermedios"

# Cargar DataFrames fuente
print("📦 Cargando DataFrames fuente desde NB01...")
dataframes = {}
for parquet_file in sorted(PERSIST_DIR.glob("raw_*.parquet")):
    nombre = parquet_file.stem.replace("raw_", "")
    dataframes[nombre] = pd.read_parquet(parquet_file)
    print(f"  ✓ {nombre}: {dataframes[nombre].shape}")

# Alias directos (igual que en NB01)
CaracteristicasGP = dataframes['caracteristicas_generales_personas']
ViviendaHog       = dataframes['viviendas_y_hogares']
GdUrbanoI         = dataframes['gastos_diarios_urbanos']
GdPersonales      = dataframes['gastos_diarios_personales_urbano']
GPersonComidasfueraH = dataframes['gastos_personales_urbano_-_comidas_preparadas_fuera_del_hogar']
GmfMP             = dataframes['gastos_menos_frecuentes_-_medio_de_pago']
GmenosFreqUrbano  = dataframes['gastos_menos_frecuentes_-_articulos']
GdComidasfueraH   = dataframes['gastos_diarios_del_hogar_urbano_-_comidas_preparadas_fuera_del_hogar']
GdUrbanoC         = dataframes['gastos_diarios_urbano_-_capitulo_c']
GdMercados        = dataframes['gastos_diarios_urbanos_-_mercados']
homologa_producto = dataframes['homologa_producto']
IPC_BASE_DIC_2018 = dataframes['cargue_inflacion']

RAW_DIR = BASE_PATH / "01_datos" / "raw"
OUTPUT_DIR = BASE_PATH / "produccion"
ARTIFACTS_DIR = OUTPUT_DIR / "modelos"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✅ NB02 listo — {len(dataframes)} fuentes disponibles")


📦 Cargando DataFrames fuente desde NB01...


  ✓ caracteristicas_generales_personas: (291590, 263)
  ✓ cargue_inflacion: (2448, 7)
  ✓ gastos_diarios_del_hogar_urbano_-_comidas_preparadas_fuera_del_hogar: (504031, 15)
  ✓ gastos_diarios_personales_urbano: (524948, 11)
  ✓ gastos_diarios_urbano_-_capitulo_c: (2035950, 9)


  ✓ gastos_diarios_urbanos: (3393520, 17)
  ✓ gastos_diarios_urbanos_-_mercados: (7569, 7)
  ✓ gastos_menos_frecuentes_-_articulos: (2111331, 12)
  ✓ gastos_menos_frecuentes_-_medio_de_pago: (87201, 134)


  ✓ gastos_personales_urbano_-_comidas_preparadas_fuera_del_hogar: (475963, 14)
  ✓ homologa_ciiu: (462, 7)
  ✓ homologa_nivel_academico: (117, 6)
  ✓ homologa_producto: (1597, 7)
  ✓ viviendas_y_hogares: (87201, 105)

✅ NB02 listo — 14 fuentes disponibles


# **2. Proceso de limpieza, homologacion y construccion de variables analíticas.**


## Nota Metodológica: Diseño Muestral ENPH 2016-2017 y Factor de Expansión FEX_C

### 1. Diseño Muestral
La **Encuesta Nacional de Presupuestos de los Hogares (ENPH) 2016-2017** del DANE utiliza un diseño **probabilístico, estratificado y por conglomerados en múltiples etapas**:

| Característica | Descripción |
|---|---|
| **Probabilístico** | Probabilidad de selección conocida y > 0 para cada unidad |
| **Estratificado** | Estratos por región, dominio urbano/rural y estrato socioeconómico |
| **Conglomerados** | UPM = segmento cartográfico; dentro: viviendas → hogares → personas |
| **Cobertura temporal** | 12 meses continuos (julio 2016 – junio 2017) |
| **Cobertura geográfica** | 24 ciudades principales, áreas metropolitanas y cabeceras municipales |

> **Implicación estadística fundamental**: los estimadores no ponderados (OLS sin FEX_C) producen resultados sesgados porque ignoran la probabilidad diferencial de selección de cada unidad. Todo análisis inferencial debe aplicar FEX_C.

---

### 2. Jerarquía de unidades: Vivienda — Hogar — Persona

```
Vivienda  (unidad física)
  └── Hogar  (grupo que comparte techo y presupuesto alimentario)
        └── Persona / Integrante  ← UNIDAD DE ANÁLISIS de este proyecto
```

**Justificación del nivel de análisis individual**:
- El objetivo del modelo es estimar el **gasto personal mensual** de un solicitante de crédito, no el gasto agregado del hogar.
- `FEX_C` está asignado a nivel de **persona** en la ENPH, lo que habilita inferencia poblacional individual.
- La variable objetivo (`log_gastos_2025`) es el gasto personal, no el gasto del hogar dividido entre integrantes.
- El scoring crediticio evalúa la **capacidad de pago individual**; modelar a nivel hogar introduciría sesgo por tamaño familiar.
- Identificador único de persona: `Id_Person = DIRECTORIO + '_' + ORDEN`.

---

### 3. Factor de Expansión FEX_C — Aplicación en el Proyecto

Un registro con `FEX_C = 1 500` representa a **1 500 personas reales** con las mismas características socioeconómicas en la población colombiana.

| Componente analítico | Mecanismo de ponderación | Resultado |
|---|---|---|
| Percentiles de ingreso (`Percentil_Ingreso_w`) | `pd.qcut` con pesos FEX_C | Deciles reflejan distribución real del país |
| Regresión WLS | `sample_weight = FEX_C_norm` en `LinearRegression` | Coeficientes = efectos **poblacionales** |
| Errores estándar | Estimador robusto `HC1` | Corrige heteroscedasticidad propia de encuestas |
| K-Means ponderado | `sample_weight = FEX_C_norm` en `KMeans.fit()` | Centroides representan el grueso de la PEA, no outliers |
| Tabla de ratios gasto/ingreso | Percentiles ponderados por cluster | Referencia válida para toda la PEA formal |
| Residuos de referencia | Mediana ponderada | `RESIDUO_GLOBAL` y `RESIDUO_INICIAL_GRUPO` son estimadores poblacionales |

**Normalización**: `FEX_C_norm = FEX_C / mean(FEX_C)` garantiza media = 1, preservando la dirección de los pesos con estabilidad numérica.

---

### 4. Garantía de Representatividad Poblacional

La representatividad se garantiza mediante tres mecanismos:

1. **Pesos en entrenamiento**: WLS minimiza la suma de errores cuadráticos **ponderados** por FEX_C → coeficientes = estimadores del efecto promedio poblacional.
2. **Validación cruzada ponderada**: CV de 5 folds con `sample_weight=FEX_C_norm` → R² y RMSE son métricas poblacionales, no muestrales.
3. **Auditoría de cobertura** (Celda siguiente): imprime cobertura muestral (n registros) y cobertura poblacional (∑FEX_C filtrado / ∑FEX_C total ENPH), verificando consistencia con el universo de inferencia declarado.

**Universo de inferencia del modelo**: Personas Económicamente Activas (PEA) con ingresos ≥ 1 SMMLV, residentes en las áreas cubiertas por la ENPH. Los resultados **no se extrapolan** a población inactiva ni a informales con ingresos < 1 SMMLV.

### 5. Propuesta de análisis descriptivo para la sustentación metodológica



Se recomienda presentar el análisis descriptivo en dos planos paralelos: **muestra observada** y **población expandida con FEX_C**. Esa comparación permite evidenciar el sesgo que aparece cuando se ignora el diseño muestral y justificar por qué los resultados finales deben reportarse con ponderación.



#### Bloque A. Cobertura, dominio de inferencia y trazabilidad del filtro



- Reportar `n` de la ENPH original, `n` del público objetivo y `∑FEX_C` en ambos conjuntos.

- Mostrar el porcentaje de cobertura muestral y el porcentaje de población expandida representada por el subconjunto modelado.

- Declarar explícitamente el **universo de inferencia**: PEA con ingreso mayor o igual a 1 SMMLV en las áreas cubiertas por la ENPH.



#### Bloque B. Composición sociodemográfica: muestra vs población



- Tablas de distribución para `sexo`, `estrato`, `actividad_ppal`, `nivel_educ_agrupado`, `DOMINIO` y grupos de edad.

- En cada tabla, presentar dos columnas: `% muestra` y `% población expandida (FEX_C)`.

- Interpretar las diferencias como evidencia de **sobrerrepresentación o subrepresentación muestral**.



#### Bloque C. Variables monetarias clave



- Comparar media, mediana, percentiles 25, 50, 75 y 90 de `ingresos_2025`, `GastoMes_2025` y `ratio_gastos_2025` con y sin ponderación.

- Mantener histogramas o densidades superpuestas para resaltar el desplazamiento de la distribución al aplicar FEX_C.

- Reportar también cortes por estrato y actividad principal para mostrar heterogeneidad estructural.



#### Bloque D. Relación entre explicativas y variable objetivo



- Graficar `log_ingresos_2025` vs `log_gastos_2025` con línea de ajuste muestral y línea de ajuste ponderada.

- Comparar gasto promedio ponderado por grupos de `estrato`, `nivel_educ_agrupado`, `actividad_ppal` y `Aportantes_Hogar`.

- Mostrar que el signo y magnitud de las asociaciones relevantes se conservan, pero su intensidad cambia cuando la estimación se lleva a nivel poblacional.



#### Bloque E. Diagnóstico del modelo y comparación metodológica



- Conservar las gráficas originales del modelo y añadir la versión ponderada para residuos, coeficientes y centroides.

- Explicar que la versión ponderada es la base inferencial; la no ponderada se deja como contraste metodológico.

- Resaltar que los coeficientes del modelo ponderado se interpretan como **efectos promedio poblacionales dentro del dominio de inferencia**, no como relaciones puramente muestrales.



#### Justificación de modelar a nivel individuo y no por hogar



1. La unidad del problema aplicado es la **persona solicitante**; la decisión de crédito se toma sobre su capacidad de pago individual.

2. `FEX_C` está definido por el DANE a nivel de registro individual, por lo que la expansión poblacional válida del estudio también es individual.

3. El gasto objetivo del proyecto es **gasto personal mensual**, no gasto agregado del hogar. Agregar por hogar mezclaría patrones de consumo de individuos con distinto rol económico.

4. Las variables del hogar y de la vivienda se incorporan como **contexto explicativo**, pero la unidad observacional y de inferencia sigue siendo la persona.

5. Trabajar por hogar implicaría redefinir la variable objetivo, recalcular pesos y cambiar el universo de inferencia; sería otro estudio.



#### Justificación de representatividad poblacional



La representatividad se sostiene porque el estimador final respeta la estructura del diseño muestral mediante el uso sistemático de `FEX_C`. En términos prácticos:



- Cada observación aporta al modelo en proporción al número de personas reales que representa.

- Las medias, percentiles, coeficientes y centroides dejan de describir solo a la muestra y pasan a describir a la población objetivo expandida.

- La auditoría de cobertura verifica que el subconjunto filtrado conserva una masa poblacional coherente dentro de la ENPH.

- La inferencia se reporta únicamente sobre el dominio para el cual los pesos siguen siendo válidos.



> Nota técnica: en este proyecto se usan errores estándar robustos HC1 como aproximación práctica. Si se dispusiera de identificadores completos de estrato y conglomerado para análisis de encuestas complejas, la siguiente mejora sería estimar varianzas con un diseño survey explícito.

El presente análisis se fundamenta en la información proveniente de dos cuadernos de datos, los cuales contienen variables sociodemográficas, laborales, de ingresos y de condiciones de la vivienda, recolectadas a nivel de personas y hogares


## Caracteristicas de vivienda y hogares


> **Nota técnica:** Las columnas numéricas se fuerzan con `pd.to_numeric` antes de operar para evitar `ZeroDivisionError` al leer desde parquet (donde tipos int pueden serializarse como float o str).

In [2]:
# Verificar que las columnas necesarias existen y son del tipo correcto
CaracteristicasGP[['DIRECTORIO', 'ORDEN']] = CaracteristicasGP[['DIRECTORIO', 'ORDEN']].astype(str)

# Crear la columna 'Id_Person' concatenando 'DIRECTORIO' y 'ORDEN' como identificador unico y llave para los otros DF
CaracteristicasGP['Id_Person'] = CaracteristicasGP['DIRECTORIO'] + "_" + CaracteristicasGP['ORDEN']

# Contar cuántas personas hay en cada hogar
resumen = CaracteristicasGP.groupby('DIRECTORIO')['ORDEN'].count().reset_index()
resumen.rename(columns={'ORDEN': 'PersonasHogar'}, inplace=True)

# Unir esta información al DataFrame original usando 'DIRECTORIO'
CaracteristicasGP = CaracteristicasGP.merge(resumen, on='DIRECTORIO', how='left')

# Renaming columns
variables_vivienda = ['DIRECTORIO','REGION','DOMINIO','P5747','P5090','P6008','P8520S1A1','P5140']

ViviendaHog_def = ViviendaHog[variables_vivienda].copy()
ViviendaHog_def = ViviendaHog_def.rename(columns={
    'DIRECTORIO': 'DIRECTORIO',
    'REGION': 'REGION',
    'DOMINIO': 'DOMINIO',
    'P5747': 'Clase_Vivienda',
    'P5090': 'Tipo_Vivienda',
    'P6008': 'PersonasHogar_informado',
    'P8520S1A1': 'Estrato',
    'P5140': 'Valor_Arriendo'
})

ViviendaHog_def['Valor_Arriendo'] = ViviendaHog_def['Valor_Arriendo'].replace({' ': '0'})
ViviendaHog_def['Valor_Arriendo'] = ViviendaHog_def['Valor_Arriendo'].astype(int)
ViviendaHog_def['DIRECTORIO'] = ViviendaHog_def['DIRECTORIO'].astype(str)

CaracteristicasGP_1_1 = CaracteristicasGP.merge(ViviendaHog_def, how="left", on="DIRECTORIO")
CaracteristicasGP_1_1.shape

# Lista de ciudades válidas
ciudades_validas = ["BARRANQUILLA","CARTAGENA","RIOHACHA","SINCELEJO","SANTA MARTA","VALLEDUPAR","MONTERÍA","SOLEDAD","BOGOTÁ",
                    "MEDELLÍN Y A.M.","MANIZALEZ Y A.M.","ARMENIA","NEIVA","FLORENCIA","IBAGUÉ","PEREIRA Y A.M.","RIONEGRO","YOPAL",
                    "ARAUCA","CÚCUTA Y A.M.","BUCARAMANGA Y A.M.","TUNJA","VILLAVICENCIO","BARRANCABERMEJA","CALI","PASTO","QUIBDÓ",
                    "YUMBO","POPAYÁN","BUENAVENTURA"]

# Aplicar las condiciones para poblar la nueva variable 'Publico_Objetivo'
CaracteristicasGP_1_1["Publico_Objetivo"] = "publico_objetivo"

CaracteristicasGP_1_1.loc[CaracteristicasGP_1_1["P6040"] < 18, "Publico_Objetivo"] = "1_menor_de_edad"
CaracteristicasGP_1_1.loc[CaracteristicasGP_1_1["P6040"] >= 80, "Publico_Objetivo"] = "2_mayor_80"
CaracteristicasGP_1_1.loc[~CaracteristicasGP_1_1["DOMINIO"].isin(ciudades_validas), "Publico_Objetivo"] = "3_no_aplica_ciudad"

publico_objetivo = CaracteristicasGP_1_1.loc[
    CaracteristicasGP_1_1['Publico_Objetivo'] == 'publico_objetivo'
].copy()
publico_objetivo.shape

publico_objetivo['P6500'] = pd.to_numeric(publico_objetivo['P6500'], errors='coerce').fillna(0).astype(int)
publico_objetivo['P7070'] = pd.to_numeric(publico_objetivo['P6500'], errors='coerce').fillna(0).astype(int)
publico_objetivo['P6760'] = pd.to_numeric(publico_objetivo['P6760'], errors='coerce').fillna(0).astype(int)

CaracteristicasGP_1_1.groupby("Publico_Objetivo", as_index=True)["DIRECTORIO"].count()

publico_objetivo['salario_1'] = publico_objetivo['P6500'].replace({98: 0, 99: 0}).fillna(0)
publico_objetivo['salario_2'] = publico_objetivo['P7070'].replace({98: 0, 99: 0}).fillna(0)

columns_to_convert = [col for col in ['P6510', 'P6510S1', 'P6510S2',
                                      'P6585S1', 'P6585S1A1', 'P6585S1A2',
                                      'P6585S2', 'P6585S2A1', 'P6585S2A2',
                                      'P6585S3', 'P6585S3A1', 'P6585S3A2',
                                      'P1653S1', 'P1653S1A1', 'P1653S1A2',
                                      'P1653S2', 'P1653S2A1', 'P1653S2A2',
                                      'P1653S3', 'P1653S3A1', 'P1653S3A2',
                                      'P1653S4', 'P1653S4A1', 'P1653S4A2',
                                      'P6779S1',
                                      'P6750', 'P6760',
                                      'P550',
                                      'P7500S2A1',
                                      'P6871',
                                      'P7500S1A1', 'P7500S4A1',
                                      'P7500S5A1'] if col in publico_objetivo]

publico_objetivo.loc[:, columns_to_convert] = (
    publico_objetivo[columns_to_convert]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(int)
 )

filtro_horas_extras = publico_objetivo[(publico_objetivo["P6510S1"] > 0)]

publico_objetivo['HorasE'] = np.where(
     (publico_objetivo['P6510S2'] == 1), 0, publico_objetivo['P6510S1'])

publico_objetivo['SubAlimentacion'] = np.where(
     (publico_objetivo['P6585S1A2'] == 1), 0, publico_objetivo['P6585S1A1'])

publico_objetivo['SubTrasp'] = np.where(
     (publico_objetivo['P6585S2A2'] == 1), 0, publico_objetivo['P6585S2A1'])

publico_objetivo['SubFamiliar'] = np.where(
     (publico_objetivo['P6585S3A2'] == 1), 0, publico_objetivo['P6585S3A1'])

publico_objetivo['PrimasT'] = np.where(
     (publico_objetivo['P1653S1A2'] == 1), 0, publico_objetivo['P1653S1A1'])

publico_objetivo['Bonifica'] = np.where(
     (publico_objetivo['P1653S2A2'] == 1), 0, publico_objetivo['P1653S2A1'])

publico_objetivo['Viaticos_asalariados'] = np.where(
     (publico_objetivo['P1653S3A2'] == 1), 0, publico_objetivo['P1653S3A1'])

publico_objetivo['Representa_asalariados'] = np.where(
     (publico_objetivo['P1653S4A2'] == 1), 0, publico_objetivo['P1653S4A1'])

publico_objetivo['Viaticos_mes'] = publico_objetivo['P6779S1']

# FIX: forzar numeric antes de dividir para evitar ZeroDivisionError
# (al leer desde parquet, columnas int pueden llegar como str o float con ceros)
_p6750 = pd.to_numeric(publico_objetivo['P6750'], errors='coerce').fillna(0)
_p6760 = pd.to_numeric(publico_objetivo['P6760'], errors='coerce').fillna(0).replace(0, np.nan)
publico_objetivo['Honorarios'] = np.where(
    _p6760.isna(), 0,
    (_p6750 / _p6760).fillna(0).astype(int)
)

_p550 = pd.to_numeric(publico_objetivo['P550'], errors='coerce').fillna(0)
publico_objetivo['Ganancias'] = np.where(
    _p550 == 0, 0,
    (_p550 / 12).fillna(0).astype(int)
)

publico_objetivo['Pension'] = publico_objetivo['P7500S2A1']

publico_objetivo['rentista_arrriendos'] = (
    publico_objetivo[['P7500S1A1', 'P7500S4A1']]
    .fillna(0)
    .sum(axis=1)
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(int)
)

publico_objetivo['rentista_vehiculos_maquinas'] = (
    publico_objetivo[['P7500S5A1']]
    .fillna(0)
    .sum(axis=1)
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(int)
)

Revision subsidio de Alimentacion para asalariados por el gobierno


In [3]:
#Empleado u obrero del gobierno:personas con ingresos hasta ($1.395.600) tienen derecho a subsidio de alimentación ($49.767),
sub_alimentos=publico_objetivo[(publico_objetivo["P6430"].isin(['2'])) & (publico_objetivo["salario_1"]<= 1395600) & (publico_objetivo["SubAlimentacion"]<= 49767)]
sub_alimentos.shape

#homologar:#Empleado u obrero del gobierno:personas con ingresos hasta ($1.395.600) tienen derecho a subsidio de alimentación ($49.767) 1 #2 » Obrero o empleado del gobierno:Empleado de la pregunta P6430
publico_objetivo['SubAlimentacion'] = np.where(
    ((publico_objetivo["P6430"].isin(['2'])) & (publico_objetivo["salario_1"]<= 1395600) &(publico_objetivo["salario_1"]> 0) & (publico_objetivo["SubAlimentacion"]<= 49767)),49767,publico_objetivo['SubAlimentacion'])

sub_alimentos=publico_objetivo[publico_objetivo["P6430"].isin(['2']) & (publico_objetivo["salario_1"]<= 1395600) & (publico_objetivo["SubAlimentacion"]< 49767)]
sub_alimentos.groupby(['SubAlimentacion','salario_1']).size().reset_index(name='conteo').sort_values(by='SubAlimentacion', ascending=True).head(10)


,SubAlimentacion,salario_1,conteo
0,0,0,214
1,300,0,1
2,35000,0,3


Revision subsidio de transporte: para asalariados por el gobierno


In [4]:
#Empleado u obrero del gobierno:Para el caso de las personas con rangos de ingresos de 1 a 2 salarios mínimos (1475434) tienen derecho a auxilio o subsidio de transporte ($74.000). Por lo tanto este tipo de subsidio debe ser incluido.
publico_objetivo['SubTrasp'] = np.where(
    ((publico_objetivo["P6430"].isin(['2'])) & (publico_objetivo["salario_1"]<= 1475434) &(publico_objetivo["salario_1"]> 0) & (publico_objetivo["SubTrasp"]<= 74000)),74000,publico_objetivo['SubTrasp'])

sub_trans=publico_objetivo[publico_objetivo["P6430"].isin(['2']) & (publico_objetivo["salario_1"]<= 1475434) & (publico_objetivo["SubTrasp"]< 74000)]
sub_trans.groupby(['SubTrasp','salario_1']).size().reset_index(name='conteo').sort_values(by='SubTrasp', ascending=True).head(10)

#¿ A cuántos meses corresponde lo que recibió? (P6760): revision de ingresos de asalariado vs variable a cuantos meses corresponde.
con_salario = publico_objetivo[publico_objetivo["salario_1"] > 0]
print(con_salario.shape)
con_salario.groupby(['P6760']).size().reset_index(name='conteo').sort_values(by='P6760', ascending=True).head(10)#debe ser cero porque el salario es mensual

salario_diario = con_salario[(con_salario["P6871"] ==1)]
salario_semanal = con_salario[(con_salario["P6871"] ==2)]


(46953, 289)


revision del valor del salario de la actividad ppal vs salario de la actividad secundaria vemos que es el mismo dato


In [5]:
mismo_salario = publico_objetivo[(publico_objetivo["salario_1"] > 0) & (publico_objetivo["salario_1"] == publico_objetivo["salario_2"])]
diferente_salario= publico_objetivo[(publico_objetivo["salario_1"] > 0) & (publico_objetivo["salario_1"] != publico_objetivo["salario_2"])]
print(mismo_salario.shape)
print(diferente_salario.shape)


(46953, 289)
(0, 289)


Ingreso mensual como empleado


In [6]:
publico_objetivo['ingreso_empleado'] = np.where(
    (publico_objetivo['P6430'].isin(['1', '2', '3','8'])),
    publico_objetivo[['salario_1', 'HorasE', 'SubAlimentacion', 'SubTrasp', 'SubFamiliar', 'PrimasT', 'Bonifica', 'Viaticos_asalariados', 'Representa_asalariados']].fillna(0).sum(axis=1),
    0
)
publico_objetivo.shape


(151575, 290)

Ingreso mensual como independiente


In [7]:
publico_objetivo['ingreso_independiente'] = np.where(
    (publico_objetivo['P6430'].isin(['4','5'])),
    publico_objetivo[['Ganancias','Honorarios','rentista_arrriendos','rentista_vehiculos_maquinas']].fillna(0).sum(axis=1),
    0
)

#¿ A cuántos meses corresponde lo que recibió? (P6760)
con_ingreso_independiente = publico_objetivo[publico_objetivo["ingreso_independiente"] > 0]
print(con_ingreso_independiente.shape)
con_ingreso_independiente.groupby(['ingreso_independiente']).size().reset_index(name='conteo').sort_values(by='ingreso_independiente', ascending=True).head(10)#no debe ser cero


(40595, 291)


,ingreso_independiente,conteo
0,8,6
1,24,1
2,98,564
3,99,321
4,196,1
5,4166,1
6,5000,10
7,5833,1
8,6000,1
9,7000,1


Ingreso mensual como pensionado: VARIABLE Pension


Ingreso mensual total


In [8]:
publico_objetivo['ingreso_mensual_total'] = publico_objetivo[['Pension', 'ingreso_empleado','ingreso_independiente']].fillna(0).sum(axis=1).apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
publico_objetivo.head(3)

# Filtrar el dataframe para las filas donde ingreso_mensual_total > 0
df_filtrado = publico_objetivo[publico_objetivo['ingreso_mensual_total'] > 0]

# Calcular la variable actividad_ppal solo para las filas filtradas
actividad_ppal_temp = np.select(
    [
        df_filtrado[['Pension', 'ingreso_empleado', 'ingreso_independiente']].idxmax(axis=1) == 'Pension',
        df_filtrado[['Pension', 'ingreso_empleado', 'ingreso_independiente']].idxmax(axis=1) == 'ingreso_empleado',
        df_filtrado[['Pension', 'ingreso_empleado', 'ingreso_independiente']].idxmax(axis=1) == 'ingreso_independiente'
    ],
    ['pensionado', 'empleado', 'independiente'],
    default='Sin categoría'
)

# Asignar los valores calculados solo a las filas filtradas
publico_objetivo.loc[publico_objetivo['ingreso_mensual_total'] > 0, 'actividad_ppal'] = actividad_ppal_temp
publico_objetivo['actividad_ppal'] = publico_objetivo['actividad_ppal'].fillna('sin_categoria')

#publico_objetivo.head(2)

#publico_objetivo.groupby(['actividad_ppal']).size().reset_index(name='conteo').sort_values(by='actividad_ppal', ascending=True).tail(10)


REVISION DE LA VARIABLE actividad_ppal


In [9]:
#realizo una copia dle dataframe para no dañarlo
publico_objetivo_actividad=publico_objetivo.copy()

#revision de minimos y maximos de ingreso para EMPLEADO
empleado= publico_objetivo_actividad[publico_objetivo_actividad['actividad_ppal'] =='empleado']
print(empleado.groupby(['ingreso_mensual_total']).size().reset_index(name='conteo').sort_values(by='ingreso_mensual_total', ascending=False).tail(5))
print(empleado.groupby(['ingreso_mensual_total']).size().reset_index(name='conteo').sort_values(by='ingreso_mensual_total', ascending=False).head(5))

#revision de minimos y maximos de ingreso para INDEPENDIENTE
independiente= publico_objetivo_actividad[publico_objetivo_actividad['actividad_ppal'] =='independiente']
print(independiente.groupby(['ingreso_mensual_total']).size().reset_index(name='conteo').sort_values(by='ingreso_mensual_total', ascending=False).tail(5))
print(independiente.groupby(['ingreso_mensual_total']).size().reset_index(name='conteo').sort_values(by='ingreso_mensual_total', ascending=False).head(5))

#revision de minimos y maximos de ingreso para PENSIONADO
pensionado= publico_objetivo_actividad[publico_objetivo_actividad['actividad_ppal'] =='pensionado']
print(pensionado.groupby(['ingreso_mensual_total']).size().reset_index(name='conteo').sort_values(by='ingreso_mensual_total', ascending=False).tail(5))
print(pensionado.groupby(['ingreso_mensual_total']).size().reset_index(name='conteo').sort_values(by='ingreso_mensual_total', ascending=False).head(5))

#revision de minimos y maximos de ingreso para sin_categoria
sin_categoria= publico_objetivo_actividad[publico_objetivo_actividad['actividad_ppal'] =='sin_categoria']
print(sin_categoria.groupby(['ingreso_mensual_total']).size().reset_index(name='conteo').sort_values(by='ingreso_mensual_total', ascending=False).tail(5))
print(sin_categoria.groupby(['ingreso_mensual_total']).size().reset_index(name='conteo').sort_values(by='ingreso_mensual_total', ascending=False).head(5))

#pensionados y tambien tienen actividad como empleado y tienen el mismo salario PREVALECE COMO PENSIONADO:
pensionados_empleado = publico_objetivo_actividad[(publico_objetivo_actividad["actividad_ppal"]=='pensionado') &(publico_objetivo_actividad["ingreso_empleado"]==publico_objetivo_actividad["Pension"])]
print(pensionados_empleado.shape)
pensionados_empleado.head(3)

#pensionados y tambien tienen actividad como independiente y tienen el mismo salario PREVALECE COMO PENSIONADO:
pensionados_independientes = publico_objetivo_actividad[(publico_objetivo_actividad["actividad_ppal"]=='pensionado') &(publico_objetivo_actividad["ingreso_independiente"]==publico_objetivo_actividad["Pension"])]
print(pensionados_independientes.shape)
pensionados_independientes.head(3)

#clasificados como empleado y tambien tienen actividad como independiente: NINGUNO
empleados = publico_objetivo_actividad[(publico_objetivo_actividad["actividad_ppal"]=='empleado')]
empleados_independiente = empleados[(empleados["ingreso_independiente"]>0) & (empleados["ingreso_empleado"]>0)]
print(empleados.shape)
print(empleados_independiente.shape)
empleados_independiente.head(3)

#clasificados como INDEPENDIENTE y tambien tienen actividad como EMPLEADO : NINGUNO
independiente = publico_objetivo_actividad[(publico_objetivo_actividad["actividad_ppal"]=='independiente')]
empleados_independiente = independiente[(independiente["ingreso_independiente"]>0) & (independiente["ingreso_empleado"]>0)]
print(independiente.shape)
print(empleados_independiente.shape)
empleados_independiente.head(3)


   ingreso_mensual_total  conteo
4                  17000       1
3                  10800       1
2                  10000       2
1                   5000       1
0                     98       3
      ingreso_mensual_total  conteo
7094               82041600       1
7093               80000000       1
7092               49800000       1
7091               44000000       1
7090               41000000       1
   ingreso_mensual_total  conteo
4                    196       1
3                     99     311
2                     98     540
1                     24       1
0                      8       6
      ingreso_mensual_total  conteo
1103              100000000       1
1102               80000000       1
1101               70000000       1
1100               50000000       2
1099               45000000       1
   ingreso_mensual_total  conteo
4                  40543       1
3                  37000       1
2                  30000       1
1                  27000       3
0      

   ingreso_mensual_total  conteo
0                      0   56213
   ingreso_mensual_total  conteo
0                      0   56213
(31, 293)
(31, 293)
(46974, 293)
(0, 293)


(39804, 293)
(0, 293)


,DIRECTORIO,SECUENCIA_ENCUESTA,SECUENCIA_P,ORDEN,P6020,P6040,P6050,P6080,P5170,P5180,P5180S1,P5180S2,P6060,P6060S1,P6081,P6081S1,P6083,P6083S1,P1650,P6071,P6071S1,P6090,P6100,P6110,P6120,P6160,P6170,P6175,P6180,P6180S1,P6180S2,P8610,P8610S1,P8610S2,P6207M1,P6207M2,P6207M3,P6207M4,P6207M5,P6207M6,P6207M7,P6207M8,P6207M9,P6207M10,P8612,P8612S1,P8612S2,P6236,P6236M1,P6236M2,P6236M3,P6236M4,P6236M5,P6236M6,P6236M7,P6236M8,P6236M9,P6236M10,P1664,P1664S1,P1664S2,P1664M1,P1664M2,P1664M3,P1664M4,P1664M5,P1664M6,P1664M7,P1664M8,P6210,P6210S1,P6210S2,P6240,P6250,P6260,P6270,P6280,P6300,P6310,P6320,P6330,P6340,P6350,P6370S1,P6390S1,P6440,P6450,P1652S1,P1652S1A1,P1652S2,P1652S2A1,P6426,P6430,P6500,P6510,P6510S1,P6510S2,P6590,P6590S1,P6600,P6600S1,P6610,P6610S1,P6620,P6620S1,P6585S1,P6585S1A1,P6585S1A2,P6585S2,P6585S2A1,P6585S2A2,P6585S3,P6585S3A1,P6585S3A2,P1653S1,P1653S1A1,P1653S1A2,P1653S2,P1653S2A1,P1653S2A2,P1653S3,P1653S3A1,P1653S3A2,P1653S4,P1653S4A1,P1653S4A2,P6630S1,P6630S1A1,P6630S2,P6630S2A1,P6630S3,P6630S3A1,P6630S4,P6630S4A1,P6630S5,P6630S5A1,P6630S6,P6630S6A1,P6640,P6640S1,P6765,P6750,P6760,P550,P6779,P6779S1,P549,P1651,P1651S1,P6790,P6800,P6850,P6830,P6870,P6871,P6880,P6920,P6920S1,P6940,P6990,P6990S1,P9450,P9450S2,P9450S1,P7040,P7045,P7050,P7070,P1665,P1665S1,P7310,P9460,P9460S1,P7422,P7422S1,P7472,P7472S1,P7500S1,P7500S1A1,P7500S4,P7500S4A1,P7500S5,P7500S5A1,P7500S2,P7500S2A1,P7500S3,P7500S3A1,P7510S1,P7510S1A1,P7510S2,P7510S2A1,P7510S3,P7510S3A1,P7510S4,P7510S4A1,P7510S5,P7510S5A1,P7510S6,P7510S6A1,P7510S10,P7510S10A1,P7510S9,P7510S9A1,P1668S1,P1668S1A2,P1668S1A1,P1668S1A3,P1668S1A4,P1668S2,P1668S2A1,P1668S2A2,P1668S2A3,P1668S2A4,P1668S3,P1668S3A1,P1668S3A2,P1668S3A3,P1668S3A4,P1668S4,P1668S4A1,P1668S4A2,P1668S4A3,P1668S4A4,P1668S5,P1668S5A1,P1668S5A2,P1668S5A3,P1668S5A4,P1668S6,P1668S6A2,P1668S6A3,P1668S6A4,P1668S6A5,P7513S1,P7513S1A1,P7513S2,P7513S2A1,P7513S3,P7513S3A1,P7513S4,P7513S4A1,P7513S5,P7513S5A1,P7513S6,P7513S6A1,P7513S7,P7513S7A1,P7513S7A2,P7513S8,P7513S8A1,P7513S9,P7513S9A1,P7513S10,P7513S10A1,P7513S11,P7513S11A1,P7513S12,P7513S12A1,P7514,P7514S1,P7516,P7516S1,FEX_C,Id_Person,PersonasHogar,REGION,DOMINIO,Clase_Vivienda,Tipo_Vivienda,PersonasHogar_informado,Estrato,Valor_Arriendo,Publico_Objetivo,salario_1,salario_2,HorasE,SubAlimentacion,SubTrasp,SubFamiliar,PrimasT,Bonifica,Viaticos_asalariados,Representa_asalariados,Viaticos_mes,Honorarios,Ganancias,Pension,rentista_arrriendos,rentista_vehiculos_maquinas,ingreso_empleado,ingreso_independiente,ingreso_mensual_total,actividad_ppal


Variable ingreso en SMMLV


In [10]:
#CREO LA VARIABLE FECHA DE LA ENCUESTA PARA CALCULAR EL INGRESO EN SMMLV Y PODER EXCLUIR POBLACION CON MENOS DE 1 SMMLV
ViviendaHog_fecha = ViviendaHog[['DIRECTORIO', 'PERIODO']].set_index('DIRECTORIO')
print(ViviendaHog_fecha.shape)
ViviendaHog_fecha.head(2)

publico_objetivo["DIRECTORIO"] = pd.to_numeric(publico_objetivo["DIRECTORIO"], errors="coerce").fillna(0).astype("int64")

publico_objetivo_2=publico_objetivo.merge(ViviendaHog_fecha, how="left", on="DIRECTORIO")
print(publico_objetivo_2.shape)
publico_objetivo_2.head(3)

publico_objetivo_2['ingreso_smmlv'] = np.where(
    publico_objetivo_2['PERIODO'].isin([201607, 201608, 201609, 201610, 201611, 201612]),
    (publico_objetivo_2['ingreso_mensual_total'] / 689454).astype(float),
    np.where(
        publico_objetivo_2['PERIODO'].isin([201701, 201702, 201703, 201704, 201705, 201706]),
        (publico_objetivo_2['ingreso_mensual_total'] / 737717).astype(float),
        0
    )
)
publico_objetivo_2.shape

mas_de_1_smmlv = publico_objetivo_2[(publico_objetivo_2["ingreso_smmlv"] >= 1.0)]
print(mas_de_1_smmlv.shape)
mas_de_1_smmlv.head(3)

max_ingreso_smmlv = publico_objetivo_2.groupby("actividad_ppal")["ingreso_smmlv"].max()
max_ingreso_smmlv

publico_objetivo_2.loc[publico_objetivo_2['ingreso_smmlv'] < 1.0, 'Publico_Objetivo'] = '4_menor_1_smmlv'

publico_objetivo_2.groupby(['Publico_Objetivo']).size().reset_index(name='conteo').sort_values(by='Publico_Objetivo', ascending=True).tail(10)

publico_objetivo_2['ingreso_smmlv_'] = publico_objetivo_2["ingreso_smmlv"].apply(lambda x:
    "i_Revisar" if pd.isnull(x) else
    "a_1_smmlv"  if  1.000000 <= float(x) <= 1.1000000 else
    "b_Entre_1.1_1.5_smmlv"  if  1.100001 <= float(x) <= 1.500000 else
    "c_Entre_1.6_2_smmlv"    if  1.500001 <= float(x) <= 2.000000 else
    "d_Entre_2.1_3_smmlv"    if  2.000001 <= float(x) <= 3.000000 else
    "e_Entre_3.1_4_smmlv"    if  3.000001 <= float(x) <= 4.000000 else
    "f_Entre_4.1_6_smmlv"    if  4.000001 <= float(x) <= 6.000000 else
    "g_Entre_6.1_10_smmlv"   if  6.000001 <= float(x) <= 10.000000 else
    "h_Mayor_10_smmlv"       if float(x) >= 10.000001 else
    "i_Revisar"
)
revisar=publico_objetivo_2[(publico_objetivo_2["ingreso_smmlv_"]=='i_Revisar') & (publico_objetivo_2["Publico_Objetivo"] == 'publico_objetivo')]
revisar.groupby(['ingreso_smmlv_']).size().reset_index(name='conteo').sort_values(by='ingreso_smmlv_', ascending=True)#.tail(10)

objetivo=publico_objetivo_2[publico_objetivo_2["Publico_Objetivo"] == 'publico_objetivo']
objetivo.groupby(['ingreso_smmlv_']).size().reset_index(name='conteo').sort_values(by='ingreso_smmlv_', ascending=True)#.tail(10)


(87201, 1)


(151575, 294)
(59783, 295)


,ingreso_smmlv_,conteo
0,a_1_smmlv,10515
1,b_Entre_1.1_1.5_smmlv,20244
2,c_Entre_1.6_2_smmlv,8374
3,d_Entre_2.1_3_smmlv,10696
4,e_Entre_3.1_4_smmlv,3426
5,f_Entre_4.1_6_smmlv,3988
6,g_Entre_6.1_10_smmlv,1705
7,h_Mayor_10_smmlv,835


Informacion de agrupacion de actividad economica


In [11]:
#cruzo la actividad del codigo ciiu
homologa_ciiu = dataframes['homologa_ciiu']
#homologa_ciiu_def = homologa_ciiu[['P6390S1','DESC_DIVISION','agrup_seccion']].set_index('P6390S1')
homologa_ciiu_def = homologa_ciiu[['P6390S1', 'DESC_DIVISION', 'agrup_seccion']].copy()
homologa_ciiu_def['P6390S1'] = homologa_ciiu_def['P6390S1'].astype(str)
homologa_ciiu_def.head(2)

publico_objetivo_3=publico_objetivo_2[(publico_objetivo_2['Publico_Objetivo'] == 'publico_objetivo')]
publico_objetivo_3.shape

publico_objetivo_3 = pd.merge(publico_objetivo_3,homologa_ciiu_def, how="left", left_on="P6390S1", right_on="P6390S1")
print(publico_objetivo_3.shape)
publico_objetivo_3.head(3)

#homologa en la variable agrup_seccion los que no tienen ciiu por 0_sin_ciiu para todas las actividades:
#publico_objetivo_3['agrup_seccion'] = publico_objetivo_3['agrup_seccion'].fillna('0_sin_ciiu')
col = 'agrup_seccion'

if str(publico_objetivo_3[col].dtype) == 'category':
    publico_objetivo_3[col] = (
        publico_objetivo_3[col]
        .cat.add_categories(['0_sin_ciiu'])
        .fillna('0_sin_ciiu')
    )
else:
    publico_objetivo_3[col] = publico_objetivo_3[col].fillna('0_sin_ciiu')
publico_objetivo_3.head(3)

publico_objetivo_3.groupby(['actividad_ppal']).size().reset_index(name='conteo').sort_values(by='actividad_ppal', ascending=True).tail(10)

#sin ciiu por actividad_ppal
sin_ciiu=publico_objetivo_3[(publico_objetivo_3["agrup_seccion"] == '0_sin_ciiu')] #& (publico_objetivo_2["actividad_ppal"] == 'independiente')]
print(sin_ciiu.shape)
sin_ciiu.head(3)#
sin_ciiu.groupby(['actividad_ppal']).size().reset_index(name='conteo').sort_values(by='actividad_ppal', ascending=True)

#Independientes CON CIIU
con_ciiu=publico_objetivo_3[(publico_objetivo_3["P6390S1"] != ' ')]# & (publico_objetivo_2["actividad_ppal"] == 'independiente')]
print(con_ciiu.shape)
con_ciiu.head(3)#
con_ciiu.groupby(['actividad_ppal']).size().reset_index(name='conteo').sort_values(by='actividad_ppal', ascending=True)


(59783, 298)
(6523, 298)
(54186, 298)


,actividad_ppal,conteo
0,empleado,36578
1,independiente,16481
2,pensionado,1127


Informacion de Nivel Académico


In [12]:
#cruzo la actividad del codigo ciiu
#importar un archivo de drive:
homologa_nivel_academico = dataframes['homologa_nivel_academico']
homologa_nivel_academico.head(3)

homologa_nivel_academico_def = homologa_nivel_academico[['LLAVE','NivelAcademico_def']]
#homologa_nivel_academico_def

publico_objetivo_3['llave_nivel_academico'] = publico_objetivo_3['P6210S1'].astype(str) + publico_objetivo_3['P6210'].astype(str) + publico_objetivo_3['P6210S2'].astype(str)

publico_objetivo_3 = pd.merge(publico_objetivo_3,homologa_nivel_academico_def, how="left", left_on="llave_nivel_academico", right_on="LLAVE")

#crear la variable aportantes en el hogar
aportantes = publico_objetivo_3.groupby(['DIRECTORIO']).size().reset_index(name='conteo').sort_values(by='conteo', ascending=True)
print(aportantes.shape)
aportantes.tail(3)

publico_objetivo_4 = pd.merge(publico_objetivo_3,aportantes, how="inner", left_on="DIRECTORIO", right_on="DIRECTORIO")
print(publico_objetivo_4.shape)
publico_objetivo_4.head(3)

publico_objetivo_4.rename(columns={'DIRECTORIO': 'DIRECTORIO',
'ORDEN': 'ORDEN',
'P6020': 'Sexo',
'P6040': 'Edad',
'P1650': 'Estado_Civil',
'P6390S1': 'CIIU',
'P6426': 'Antigüedad_Actividad',
'conteo': 'Aportantes_Hogar'
}, inplace=True)

publico_objetivo_4['DIRECTORIO'] = publico_objetivo_4['DIRECTORIO'].astype(str)

print(publico_objetivo_4.shape)


(42791, 2)


(59783, 302)
(59783, 302)


### Público objetivo


In [13]:
publico_objetivo_5 = publico_objetivo_4[['DIRECTORIO', 'ORDEN', 'Id_Person', 'Sexo', 'Edad', 'Estado_Civil', 'Estrato', 'PersonasHogar', 'REGION',
                                         'DOMINIO', 'Clase_Vivienda', 'Tipo_Vivienda', 'PersonasHogar_informado', 'PERIODO', 'ingreso_smmlv',
                                         'NivelAcademico_def', 'CIIU', 'DESC_DIVISION', 'agrup_seccion', 'Antigüedad_Actividad','ingreso_smmlv_',
                                         'ingreso_mensual_total', 'actividad_ppal','Aportantes_Hogar', 'FEX_C']].copy()

#publico_objetivo_4.to_csv('/content/drive/My Drive/ArchivosDatos/poblacion_objetivo_final_300_variables.csv', sep=';', index=False)
#exportar un dataframe a drive toca eliminar variable sporuqe se reinicia

#excel_file_path = '/content/drive/My Drive/ArchivosDatos/poblacion_objetivo_final.xlsx'

# Use to_excel to save the DataFrame to Excel
#publico_objetivo_5.to_excel(excel_file_path, index=False)  # Set index=False to avoid writing row index
print(publico_objetivo_5.shape)


(59783, 25)


### Auditoría de cobertura — sustenta la Sección 5.1.4 del manuscrito

La siguiente celda audita la cobertura del filtro de público objetivo (aportantes con ingreso ≥ 1 SMMLV) respecto a la muestra completa de la ENPH 2016-2017. Las cifras producidas por esta celda se reproducen en la **Sub-sección 5.1.4 "Cobertura del público objetivo"** del manuscrito y se utilizan para responder a la observación del jurado sobre el alcance inferencial del modelo.

En particular, esta auditoría documenta:

- La cobertura muestral: porcentaje de la ENPH cruda que entra al modelo.
- La cobertura poblacional: total expandido del público objetivo respecto a la población total representada por la ENPH.
- La distribución del FEX_C en el público objetivo (mínimo, cuartiles, máximo), que evidencia la heterogeneidad de los pesos y la importancia de incorporarlos en cualquier estimación poblacional.
- Una comparación demográfica original vs. filtrado para variables clave (Sexo, Estrato), que muestra cómo el filtro de ≥ 1 SMMLV altera la composición.

**Output esperado**: cifras que sirven de soporte directo a la Tabla 1.1 y Tabla 1.2 de la Sección 5.1.4 del manuscrito.

In [14]:
# ============================================================
# VALIDACIÓN DE REPRESENTATIVIDAD POBLACIONAL — FEX_C
# ============================================================
# Audita la cobertura del filtro aplicado respecto a la muestra
# original de la ENPH y documenta qué población representa el modelo.

import numpy as np

def fex_to_numeric(series):
    """Convierte FEX_C (puede tener coma decimal) a float."""
    return pd.to_numeric(series.astype(str).str.replace(',', '.', regex=False), errors='coerce').fillna(0)

# ── Tamaños de muestra ──────────────────────────────────────────
n_original   = len(CaracteristicasGP)          # ENPH completa
n_filtrado   = len(publico_objetivo_5)          # activos ≥1 SMMLV
pct_cobertura = n_filtrado / n_original * 100

# ── Expansión poblacional (suma de FEX_C) ───────────────────────
fex_orig     = fex_to_numeric(CaracteristicasGP['FEX_C'])
fex_filtrado = fex_to_numeric(publico_objetivo_5['FEX_C'])

pob_total     = fex_orig.sum()        # población total representada por la ENPH
pob_objetivo  = fex_filtrado.sum()    # población activa ≥1 SMMLV representada
pct_pob       = pob_objetivo / pob_total * 100

print('=' * 62)
print('AUDITORÍA DE COBERTURA — DISEÑO MUESTRAL ENPH')
print('=' * 62)
print(f'  Registros ENPH total (muestra cruda)  : {n_original:>10,}')
print(f'  Registros público objetivo (≥1 SMMLV): {n_filtrado:>10,}')
print(f'  Cobertura muestral                    : {pct_cobertura:>9.1f}%')
print()
print(f'  Población total representada (FEX_C)  : {pob_total:>16,.0f}')
print(f'  Población objetivo representada       : {pob_objetivo:>16,.0f}')
print(f'  Cobertura poblacional                 : {pct_pob:>9.1f}%')
print()
print('CRITERIO: El modelo es válido para la subpoblación económicamente')
print('activa con ingreso ≥1 SMMLV. Esta es la población objetivo del')
print('scoring crediticio. Los FEX_C preservan representatividad DENTRO')
print('de esta subpoblación.')
print()

# ── Comparación demográfica: muestra original vs filtrada ───────
compare_vars = ['Sexo', 'Estrato']
print('-' * 62)
print('COMPARACIÓN DEMOGRÁFICA: original vs público objetivo')
print('-' * 62)

for var in compare_vars:
    if var in CaracteristicasGP.columns and var in publico_objetivo_5.columns:
        orig_dist = CaracteristicasGP[var].value_counts(normalize=True).sort_index()
        filt_dist = publico_objetivo_5[var].value_counts(normalize=True).sort_index()
        print(f'\n  {var}:')
        comp = pd.DataFrame({'original_%': orig_dist*100, 'filtrado_%': filt_dist*100})
        comp['diferencia_pp'] = comp['filtrado_%'] - comp['original_%']
        print(comp.round(1).to_string())

# ── Rango de FEX_C en el conjunto filtrado ──────────────────────
print(f'\n  FEX_C (público objetivo):')
print(f'  min={fex_filtrado.min():,.1f}  p25={fex_filtrado.quantile(.25):,.1f}  '
      f'p50={fex_filtrado.median():,.1f}  p75={fex_filtrado.quantile(.75):,.1f}  max={fex_filtrado.max():,.1f}')
print(f'  Suma total: {fex_filtrado.sum():,.0f} personas representadas')
print('=' * 62)


AUDITORÍA DE COBERTURA — DISEÑO MUESTRAL ENPH
  Registros ENPH total (muestra cruda)  :    291,590
  Registros público objetivo (≥1 SMMLV):     59,783
  Cobertura muestral                    :      20.5%

  Población total representada (FEX_C)  :       48,018,203
  Población objetivo representada       :        8,153,737
  Cobertura poblacional                 :      17.0%

CRITERIO: El modelo es válido para la subpoblación económicamente
activa con ingreso ≥1 SMMLV. Esta es la población objetivo del
scoring crediticio. Los FEX_C preservan representatividad DENTRO
de esta subpoblación.

--------------------------------------------------------------
COMPARACIÓN DEMOGRÁFICA: original vs público objetivo
--------------------------------------------------------------

  FEX_C (público objetivo):
  min=1.2  p25=29.9  p50=60.1  p75=121.3  max=4,139.6
  Suma total: 8,153,737 personas representadas


A continuación el tratamiento para los datos correspondientes a los gastos de los integrantes de hogar, algunas funciones son creadas teneindo en cuenta la metodología reportada por el DANE. En el siguiente link se encuentra la documentacipon tecnica soporte del cálculo:  https://microdatos.dane.gov.co/index.php/catalog/566/related-materials


## Gastos Diarios Urbano


In [15]:
print(GdUrbanoI.shape)
GdUrbanoI.columns

GdUrbano = GdUrbanoI[GdUrbanoI['NH_CGDU_P5'] == 1]#filtra para que la fomra de adquisición sea solo compra
print(GdUrbano.shape)
GdUrbano.head()

poblacion_objetivo_final = publico_objetivo_5
poblacion_objetivo_final.shape

# Verificar que las columnas necesarias existen y son del tipo correcto
GdUrbano[['DIRECTORIO', 'ORDEN']] = GdUrbano[['DIRECTORIO', 'ORDEN']].astype(str)

# Crear la columna 'Id_Person' concatenando 'DIRECTORIO' y 'ORDEN'
GdUrbano['Id_Person'] = GdUrbano['DIRECTORIO'] + "_" + GdUrbano['ORDEN']

# Calcular 'Periodicidad'
GdUrbano['Freq'] = GdUrbano['NH_CGDU_P9']

# Calcular ingreso Mensual
def calcular_gasto_mensual(df):

    # Aplicar la lógica de multiplicación según P6871
    multiplicadores = {1: 2.14, 2: 2.14, 3: 2.14, 4: 2, 5: 1, 6: 0.5, 7: (1/3), 9: 1}
    df['ValorMes'] = df['NH_CGDU_P8'] * df['NH_CGDU_P9'].map(multiplicadores)

    # Agrupar por Id_Person and sumar los valores
    # Instead of reassigning CaracteristicasGP, merge the results back
    ingreso_mes_por_persona = df.groupby('Id_Person', as_index=False)['ValorMes'].sum()
    return df

# Llamar a la función para calcular y agregar la columna 'IngresoMes'
GdUrbano = calcular_gasto_mensual(GdUrbano)

# Convertir a string y completar con ceros a la izquierda hasta 8 caracteres
GdUrbano['NH_CGDU_P1'] = GdUrbano['NH_CGDU_P1'].astype(str).str.zfill(8)

# Crear la columna DIVISIÓN con los dos primeros caracteres de NH_CGDU_P1
GdUrbano['DIVISIÓN'] = GdUrbano['NH_CGDU_P1'].str[:2]

GdUrbano.tail(3)

# Agrupar por DIRECTORIO, Id_Person y DIVISIÓN y sumar ValorMes
df_resumen = GdUrbano.groupby(['DIRECTORIO', 'Id_Person', 'DIVISIÓN'], as_index=False)['ValorMes'].sum()
print(df_resumen.shape)
df_resumen.head(3)

df_final = df_resumen.merge(
    poblacion_objetivo_final[['Id_Person']],  # solo la columna Id_Person
    on='Id_Person',
    how='inner'  # conserva solo los Id_Person presentes en poblacion_objetivo_final
)

# Verificar resultado
print(df_final.shape)
print(df_final.head())


(3393520, 17)


(3209896, 17)


(778445, 4)
(75731, 4)
  DIRECTORIO Id_Person DIVISIÓN  ValorMes
0     118277  118277_1       01   64172.0
1     118279  118279_1       01    3638.0
2     118282  118282_1       01   95952.0
3     118285  118285_2       01   60978.0
4     118286  118286_2       01   94224.0


## Gastos Diarios Urbano Capitulo C


In [16]:
print(GdUrbanoC.shape)
#GdUrbanoC = GdUrbanoCI[GdUrbanoCI['NC2_CC_P3_S2'] == 1]#¿Comprará este alimento o grupo de alimentos durante los 14 días que dura la encuesta? = 1 si
#print(GdUrbanoC.shape)
#GdUrbanoC.head(3)

# Verificar que las columnas necesarias existen y son del tipo correcto
GdUrbanoC[['DIRECTORIO', 'ORDEN']] = GdUrbanoC[['DIRECTORIO', 'ORDEN']].astype(str)

# Crear la columna 'Id_Person' concatenando 'DIRECTORIO' y 'ORDEN'
GdUrbanoC['Id_Person'] = GdUrbanoC['DIRECTORIO'] + "_" + GdUrbanoC['ORDEN']
print(GdUrbanoC.shape)
GdUrbanoC.head(3)

# Calcular 'Periodicidad'
GdUrbanoC['Freq'] = GdUrbanoC['NC2_CC_P2']

# Calcular ingreso Mensual
def calcular_gasto_mensual(df):

    # Aplicar la lógica de multiplicación según P6871
    multiplicadores = {1: 2.14, 2: 2.14, 3: 2.14, 4: 2, 5: 1, 6: 0.5, 7: (1/3), 9: 1, 11:0}
    df['ValorMes'] = df['NC2_CC_P3_S1'] * df['NC2_CC_P2'].map(multiplicadores)

    # Agrupar por Id_Person and sumar los valores
    # Instead of reassigning CaracteristicasGP, merge the results back
    ingreso_mes_por_persona = df.groupby('Id_Person', as_index=False)['ValorMes'].sum()
    return df

# Llamar a la función para calcular y agregar la columna 'IngresoMes'
GdUrbanoC = calcular_gasto_mensual(GdUrbanoC)
print(GdUrbanoC.shape)
GdUrbanoC.head(3)

# Agrupar por DIRECTORIO, Id_Person y DIVISIÓN y sumar ValorMes
df_resumenC = GdUrbanoC.groupby(['DIRECTORIO', 'Id_Person'], as_index=False)['ValorMes'].sum()
df_resumenC['DIVISIÓN'] = '01'
print(df_resumenC.shape)
df_resumenC.head(3)

# Realizar el join (inner merge) usando la columna Id_Person
df_resumenCO = df_resumenC.merge(
    poblacion_objetivo_final[['Id_Person']],  # solo la columna Id_Person
    on='Id_Person',
    how='inner'  # conserva solo los Id_Person presentes en poblacion_objetivo_final
)
# Verificar resultado
#print(df_resumenCO.shape)
#print(df_resumenCO.head())

df_resumenCO_ = df_resumenCO[
    (df_resumenCO[['ValorMes']].fillna(0).sum(axis=1)) != 0
]
print(df_resumenCO_.shape)


(2035950, 9)


(2035950, 10)


(2035950, 12)


(2035950, 4)


(7722, 4)


### concatenado de gastos diarios urbano y gastos diarios urbano capitulo C


In [17]:
df_nuevo = pd.concat([df_final, df_resumenCO_], ignore_index=True)
#print(df_nuevo.shape)
#df_nuevo.head(2)

df_GdUrbano = df_final.pivot_table(
    index=['DIRECTORIO', 'Id_Person'],
    columns='DIVISIÓN',
    values='ValorMes',
    aggfunc='sum'
).reset_index()
#df_GdUrbano.head(3)

#df_GdUrbano.groupby(['Id_Person']).size().reset_index(name='conteo').sort_values(by='conteo', ascending=False).tail(3)


### DF gastos diarios urbano: df_GdUrbano


In [18]:
print(df_GdUrbano.shape)

#df_GdUrbano.to_excel('/content/drive/My Drive/ArchivosDatos/df_GdUrbano.xlsx', index=False)
#df_GdUrbano


(57183, 10)


## Gastos menos frecuentes artículos


In [19]:
print(poblacion_objetivo_final.shape)

poblacion_objetivo_final['Percentil_Ingreso'] = pd.qcut(poblacion_objetivo_final['ingreso_mensual_total'], q=10, labels=[f"P{int(i*10)}-{int((i+1)*10)}" for i in range(10)])

# Percentil_Ingreso ponderado por FEX_C (usando quantiles poblacionales)
fex_c_perc = pd.to_numeric(
    poblacion_objetivo_final['FEX_C'].astype(str).str.replace(',', '.', regex=False),
    errors='coerce'
).fillna(1.0)
# Calcular cuantiles ponderados manualmente
_ing = poblacion_objetivo_final['ingreso_mensual_total'].values
_w   = fex_c_perc.values
_sorted_idx = _ing.argsort()
_ing_s, _w_s = _ing[_sorted_idx], _w[_sorted_idx]
_cumw = _w_s.cumsum() / _w_s.sum()
_decile_cuts_w = [_ing_s[(_cumw >= q).argmax()] for q in [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]]
_bins_w = [-float('inf')] + _decile_cuts_w + [float('inf')]
_labels_w = ['P0-10','P10-20','P20-30','P30-40','P40-50','P50-60','P60-70','P70-80','P80-90','P90-100']
poblacion_objetivo_final['Percentil_Ingreso_w'] = pd.cut(
    poblacion_objetivo_final['ingreso_mensual_total'],
    bins=_bins_w, labels=_labels_w, include_lowest=True
)
print('Percentil_Ingreso_w (ponderado FEX_C):',
      poblacion_objetivo_final['Percentil_Ingreso_w'].value_counts().sort_index())

# Verificar y convertir la columna "P10270" en GmenosFreqUrbano a int64
GmenosFreqUrbano["P10270"] = pd.to_numeric(GmenosFreqUrbano["P10270"], errors="coerce").fillna(0).astype("int64")

# Verificar y convertir la columna "llave" en homologa_producto a int64
homologa_producto["llave"] = pd.to_numeric(homologa_producto["llave"], errors="coerce").fillna(0).astype("int64")

# Realizar el merge después de la conversión
GmenosFreqUrbano_homo = pd.merge(GmenosFreqUrbano,homologa_producto,how="left",left_on="P10270",right_on="llave")
GdPersonales_homo = pd.merge(GdPersonales, homologa_producto, how="left", left_on="NC4_CC_P1_1", right_on="llave")

# Lista de DataFrames a procesar
dataframes = [df_GdUrbano,GmenosFreqUrbano_homo, GdPersonales_homo, GdComidasfueraH, GdMercados, GmfMP, GPersonComidasfueraH,poblacion_objetivo_final]

# Lista de valores a eliminar de la columna "DIRECTORIO" PORQUE EL ORDEN DE LA TABLA  CaracteristicasGP NO CUADRA con la variable cantidad de personas del hogar tabla ViviendaHog P6008
valores_a_eliminar = ['119286', '233790', '295222', '440454', '571201']
#valores_a_eliminar = [119286, 233790, 295222, 440454, 571201]

# Iterar sobre los DataFrames y eliminar las filas correspondientes
dataframes_filtrados = []
for i, df in enumerate(dataframes):
    if "DIRECTORIO" in df.columns:  # Verificar si la columna existe
        df_filtrado = df[~df["DIRECTORIO"].isin(valores_a_eliminar)]
        dataframes_filtrados.append(df_filtrado)
    else:
        print(f"La columna 'DIRECTORIO' no existe en el DataFrame {i + 1}. Continuando...")
        dataframes_filtrados.append(df)  # Si no existe, agregar el DataFrame sin cambios

# Si es necesario, reasignar los DataFrames filtrados a sus variables originales
(df_GdUrbano,GmenosFreqUrbano_homo, GdPersonales_homo, GdComidasfueraH, GdMercados, GmfMP, GPersonComidasfueraH,poblacion_objetivo_final) = dataframes_filtrados

# Filtrar un valor específico (ejemplo: DIRECTORIO igual a 233790) del DataFrame GdUrbanoC_homo
#GdPersonales_homo_filtro = GdPersonales_homo[GdPersonales_homo["DIRECTORIO"] == "233790"]

# Mostrar el resultado
#GdPersonales_homo_filtro

Id_Person_objetivo=poblacion_objetivo_final[["Id_Person"]].copy()
#print(Id_Person_objetivo.shape)
#Id_Person_objetivo.head(3)

DIRECTORIO_objetivo = poblacion_objetivo_final.groupby("DIRECTORIO", as_index=False)["Aportantes_Hogar"].count().sort_values(by='Aportantes_Hogar', ascending=True)
#print(DIRECTORIO_objetivo.shape)
#DIRECTORIO_objetivo.tail(3)

# Crear la columna 'Id_Person' concatenando 'DIRECTORIO' y 'ORDEN'
#import numpy as np
GdPersonales_homo['Id_Person'] = GdPersonales_homo['DIRECTORIO'].astype(np.str_) + "_" + GdPersonales_homo['ORDEN'].astype(np.str_)

# Mostrar el resultado
#compras_gdpersonales_homo_filtrado.head(3)

gdpersonales_homo_objetivo = pd.merge(Id_Person_objetivo,GdPersonales_homo, how="inner", left_on="Id_Person", right_on="Id_Person")
#print(gdpersonales_homo_objetivo.shape)
#gdpersonales_homo_objetivo.head(3)

#Agrupar después del filtro: NC4_CC_P6: Frecuencia de Compra: 1 » 1. Diario2 » 2.1. Varias veces3 » 3. Semanal4 » 4. Quincenal5 » 5. Mensual6 » 6. Bimestral7 » 7. Trimestral9 » 9. Esporádica por semana
agrup_freq_compras_gdpersonales_homo = gdpersonales_homo_objetivo.groupby("NC4_CC_P6", as_index=False)["DIRECTORIO"].count()
#print(agrup_freq_compras_gdpersonales_homo.shape)
#agrup_freq_compras_gdpersonales_homo.head(11)

# Convertir las columnas a string antes de aplicar .str.replace
gdpersonales_homo_objetivo["NC4_CC_P2"] = gdpersonales_homo_objetivo["NC4_CC_P2"].astype(str)
gdpersonales_homo_objetivo["NC4_CC_P5"] = gdpersonales_homo_objetivo["NC4_CC_P5"].astype(str)

# Reemplazar comas por puntos y convertir a números
gdpersonales_homo_objetivo["NC4_CC_P2"] = gdpersonales_homo_objetivo["NC4_CC_P2"].str.replace(",", ".").astype(float).astype("int64")
gdpersonales_homo_objetivo["NC4_CC_P5"] = gdpersonales_homo_objetivo["NC4_CC_P5"].str.replace(",", ".").astype(float).astype("int64")

# Verificar los cambios
#print(gdpersonales_homo_objetivo.shape)
#print(gdpersonales_homo_objetivo.dtypes)
#gdpersonales_homo_objetivo.head(3)

# Verificar y convertir la columna "P10270" en GmenosFreqUrbano a int64
GmenosFreqUrbano["P10270"] = pd.to_numeric(GmenosFreqUrbano["P10270"], errors="coerce").fillna(0).astype("int64")

# Verificar y convertir la columna "llave" en homologa_producto a int64
homologa_producto["llave"] = pd.to_numeric(homologa_producto["llave"], errors="coerce").fillna(0).astype("int64")

# Realizar el merge después de la conversión
GmenosFreqUrbano_homo = pd.merge(    GmenosFreqUrbano,    homologa_producto,    how="left",    left_on="P10270",    right_on="llave")

#frecuencia de compra: 3 » 3. Semanal 4 » 4. Quincenal 5 » 5. Mensual 6 » 6. Bimestral 7 » 7. Trimestral 8 » 8. Anual 9 » 9. Esporádica 10 » 10. Semestral

#GmenosFreqUrbano_homo.groupby("P10270S3", as_index=True)["P10270S3"].count()

#toma la base y filtra solo por los que se adquirireron en forma de compra:
GmenosFreqUrbano_homo_1=GmenosFreqUrbano_homo[GmenosFreqUrbano_homo["FORMA"]==1]
#print(GmenosFreqUrbano_homo_1.shape)
#GmenosFreqUrbano_homo_1.head(3)

tabla_3 = GmenosFreqUrbano_homo_1[GmenosFreqUrbano_homo_1["CAP"].isin(['A11', 'B11', 'C11', 'E11', 'F11', 'H11', 'J11', 'K11', 'L11'])]
#print(tabla_3.shape)
#tabla_3.groupby(['P10270S3'], dropna=False).size().reset_index(name='conteo').sort_values(by='P10270S3', ascending=False)

# Definir la función con la conversión a int64

def calcular_GmenosFreqUrbano_mes_tabla_3(row):
      # Convertir las variables NC4_CC_P2 y NC4_CC_P5 a int64
    row["P10270S3"] = float(row["P10270S3"])  # Asegurar que sea entero
    row["VALOR"] = float(row["VALOR"])  # Asegurar que sea entero

    if pd.isnull(row["P10270S3"]): #NULOS
        return row["VALOR"] * 1
    elif row["P10270S3"] == 3.0:#semanal
        return row["VALOR"] * 1
    elif row["P10270S3"] == 4.0:#quincenal
        return row["VALOR"] *  1
    elif row["P10270S3"] == 5.0:#mensual
        return row["VALOR"] *  1
    elif row["P10270S3"] == 6.0:#bimensual
        return row["VALOR"] /  2
    elif row["P10270S3"] == 7.0:#trimestral
        return row["VALOR"] /  3
    elif row["P10270S3"] == 8.0:#anual
        return row["VALOR"] / 12
    elif row["P10270S3"] == 9.0:#esporadico
        return row["VALOR"] * 1
    elif row["P10270S3"] == 10.0:#semestral
        return row["VALOR"] / 6

# Aplicar la función fila por fila
#GdUrbanos_2["GASTO_MENSUAL"] = GdUrbanos_2.apply(calcular_gdpersonales_mes, axis=1)

tabla_4 = GmenosFreqUrbano_homo_1[GmenosFreqUrbano_homo_1["CAP"].isin(['D11', 'D12', 'D13', 'D14','D15', 'D16', 'F12', 'G11', 'I11', 'J12'])]
print(tabla_4.shape)
tabla_4.groupby(['P10270S3'], dropna=False).size().reset_index(name='conteo').sort_values(by='P10270S3', ascending=False)

# Definir la función con la conversión a int64

def calcular_GmenosFreqUrbano_mes_tabla_4(row):
      # Convertir las variables NC4_CC_P2 y NC4_CC_P5 a int64
    row["P10270S3"] = float(row["P10270S3"])  # Asegurar que sea entero
    row["VALOR"] = float(row["VALOR"])  # Asegurar que sea entero

    if pd.isnull(row["P10270S3"]): #NULOS
        return row["VALOR"] * 1
    elif row["P10270S3"] == 3.0:#semanal
        return row["VALOR"] /3
    elif row["P10270S3"] == 4.0:#quincenal
        return row["VALOR"] /3
    elif row["P10270S3"] == 5.0:#mensual
        return row["VALOR"] /3
    elif row["P10270S3"] == 6.0:#bimensual
        return row["VALOR"] /  3
    elif row["P10270S3"] == 7.0:#trimestral
        return row["VALOR"] *1
    elif row["P10270S3"] == 8.0:#anual
        return row["VALOR"] / 4
    elif row["P10270S3"] == 9.0:#esporadico
        return row["VALOR"] * 1
    elif row["P10270S3"] == 10.0:#semestral
        return row["VALOR"] / 2

# Aplicar la función fila por fila
#GdUrbanos_2["GASTO_MENSUAL"] = GdUrbanos_2.apply(calcular_gdpersonales_mes, axis=1)

#TODOS DIVIDIDOS EN 12
tabla_5 = GmenosFreqUrbano_homo_1[GmenosFreqUrbano_homo_1["CAP"].isin(['B12', 'C12', 'E12', 'F13','G12', 'G121', 'G122', 'H12', 'I12', 'J13', 'J131', 'J132', 'L12'])]
print(tabla_5.shape)
tabla_5.groupby(['P10270S3'], dropna=False).size().reset_index(name='conteo').sort_values(by='P10270S3', ascending=False)

#elimina registros que no estan reportando valor de gasto son 115 registros
GmenosFreqUrbano_homo_2 = GmenosFreqUrbano_homo_1.dropna(subset=["VALOR"])

registros_iniciales = GmenosFreqUrbano_homo_1.shape[0]
registros_finales = GmenosFreqUrbano_homo_2.shape[0]

registros_eliminados = registros_iniciales - registros_finales

print("Registros iniciales:", registros_iniciales)
print("Registros finales:", registros_finales)
print("Registros eliminados:", registros_eliminados)

#elimina registros que no estan reportando valor de gasto son 115 registros
GmenosFreqUrbano_homo_1 = GmenosFreqUrbano_homo_1.dropna(subset=["VALOR"])

GmenosFreqUrbano_homo_1['Id_Person'] = GmenosFreqUrbano_homo_1['DIRECTORIO'].astype(np.str_) + "_" + GmenosFreqUrbano_homo['ORDEN'].astype(np.str_)
# Mostrar el resultado
print(GmenosFreqUrbano_homo_1.shape)
GmenosFreqUrbano_homo_1.head(3)

#compras_gdpersonales_homo_filtrado.head(3)
GmenosFreqUrbano_homo_objetivo = pd.merge(GmenosFreqUrbano_homo_1,Id_Person_objetivo, how="inner", left_on="Id_Person", right_on="Id_Person")
# Agrupar después del filtro: Frecuencia de Compra: 1 » 1. Diario2 » 2.1. Varias veces3 » 3. Semanal4 » 4. Quincenal5 » 5. Mensual6 » 6. Bimestral7 » 7. Trimestral9 » 9. Esporádica por semana
print(GmenosFreqUrbano_homo_objetivo.shape)
GmenosFreqUrbano_homo_objetivo.head(3)

# Reemplazar valores NaN por 0
GmenosFreqUrbano_homo_objetivo["VALOR"] = GmenosFreqUrbano_homo_objetivo["VALOR"].fillna(0)

# Convertir la variable "VALOR" a entero
GmenosFreqUrbano_homo_objetivo["VALOR"] = GmenosFreqUrbano_homo_objetivo["VALOR"].astype(int)

# Función para calcular GmenosFreqUrbano_mes
def calcular_GmenosFreqUrbano_mes(row):
    if row["CAP"] in ['A11', 'B11', 'C11', 'E11', 'F11', 'H11', 'J11', 'K11', 'L11']:
        return calcular_GmenosFreqUrbano_mes_tabla_3(row)
    elif row["CAP"] in ['D11', 'D12', 'D13', 'D14', 'D15', 'D16', 'F12', 'G11', 'I11', 'J12']:
        return calcular_GmenosFreqUrbano_mes_tabla_4(row)
    else:
        return row["VALOR"] / 12

# Aplicar la función a cada fila
GmenosFreqUrbano_homo_objetivo["GmenosFreqUrbano_mes"] = GmenosFreqUrbano_homo_objetivo.apply(calcular_GmenosFreqUrbano_mes, axis=1)

# Mostrar resultados
print(GmenosFreqUrbano_homo_objetivo.shape)
GmenosFreqUrbano_homo_objetivo.tail(3)

GmenosFreqUrbano_homo_objetivo[GmenosFreqUrbano_homo_objetivo["GmenosFreqUrbano_mes"]!=GmenosFreqUrbano_homo_objetivo["VALOR"]].count()

GmenosFreqUrbano_homo_objetivo[GmenosFreqUrbano_homo_objetivo["P10270S3"].isnull()].head(3)

GmenosFreqUrbano_homo_objetivo_agrup = GmenosFreqUrbano_homo_objetivo.groupby(["Id_Person"], as_index=True)["Id_Person"].count().sort_values(ascending=False)
GmenosFreqUrbano_homo_objetivo_agrup.tail(3)

#GmenosFreqUrbano_homo_objetivo[(GmenosFreqUrbano_homo_objetivo["Id_Person"] == '233205_2')]

#GmenosFreqUrbano_homo_objetivo.groupby("FORMA", as_index=False)["Id_Person"].count()

#gastos_objetivo_filtrado = GmenosFreqUrbano_homo_objetivo[(GmenosFreqUrbano_homo_objetivo["DIVISIÓN"] == 11.0)]
#gastos_objetivo_filtrado.head(10)

#Agrupacion de los gastos menos frecuentes por el campo division
# Agrupar por DIRECTORIO, Id_Person y DIVISIÓN y sumar ValorMes
df_resumen = GmenosFreqUrbano_homo_objetivo.groupby(['DIRECTORIO', 'Id_Person', 'DIVISIÓN'], as_index=False)['GmenosFreqUrbano_mes'].sum()
df_resumen.head(3)

print(df_resumen.shape)

GmenosFreqUrbano_homo_objetivo_def = df_resumen.pivot_table(
    index=['DIRECTORIO', 'Id_Person'],
    columns='DIVISIÓN',
    values='GmenosFreqUrbano_mes',
    aggfunc='sum'
).reset_index()
print(GmenosFreqUrbano_homo_objetivo_def.shape)
GmenosFreqUrbano_homo_objetivo_def.head(5)

GmenosFreqUrbano_homo_objetivo_def.columns = GmenosFreqUrbano_homo_objetivo_def.columns.astype(str)
GmenosFreqUrbano_homo_objetivo_def.rename(columns={
    "1.0": "01_menos_frc", "2.0": "02_menos_frc", "3.0": "03_menos_frc", "4.0": "04_menos_frc",
    "5.0": "05_menos_frc", "6.0": "06_menos_frc", "7.0": "07_menos_frc", "8.0": "08_menos_frc",
    "9.0": "09_menos_frc", "10.0": "10_menos_frc", "11.0": "11_menos_frc", "12.0": "12_menos_frc"
}, inplace=True)


(59783, 25)
Percentil_Ingreso_w (ponderado FEX_C): Percentil_Ingreso_w
P0-10      6540
P10-20     7746
P20-30     4520
P30-40     6051
P40-50     5598
P50-60     5879
P60-70     6039
P70-80     6689
P80-90     5655
P90-100    5066
Name: count, dtype: int64


(346259, 19)
(256871, 19)
Registros iniciales: 2017165
Registros finales: 2017050
Registros eliminados: 115


(2017050, 20)


(57836, 20)


(57836, 21)
(57836, 4)


(57836, 13)


### DF gastos menos frecuente urbano: GmenosFreqUrbano_homo_objetivo_def


In [20]:
print(GmenosFreqUrbano_homo_objetivo_def.shape)
GmenosFreqUrbano_homo_objetivo_def.head(3)

#excel_file_path = '/content/drive/My Drive/ArchivosDatos/GmenosFreqUrbano_homo_objetivo_def.xlsx'
#GmenosFreqUrbano_homo_objetivo_def.to_excel(excel_file_path, index=False)  # Set index=False to avoid writing row index


(57836, 13)


DIVISIÓN,DIRECTORIO,Id_Person,01_menos_frc,02_menos_frc,03_menos_frc,04_menos_frc,05_menos_frc,06_menos_frc,07_menos_frc,08_menos_frc,09_menos_frc,10_menos_frc,12_menos_frc
0,118277,118277_1,NaN,NaN,NaN,NaN,1700.0,NaN,NaN,NaN,NaN,NaN,NaN
1,118279,118279_1,NaN,NaN,NaN,NaN,5000.0,NaN,NaN,NaN,NaN,NaN,NaN
2,118280,118280_1,NaN,NaN,NaN,NaN,3000.0,NaN,NaN,NaN,NaN,NaN,NaN


## Gastos diarios personales urbano: GdPersonales_homo


In [21]:
# Crear la columna 'Id_Person' concatenando 'DIRECTORIO' y 'ORDEN'
#import numpy as np
GdPersonales_homo['Id_Person'] = GdPersonales_homo['DIRECTORIO'].astype(np.str_) + "_" + GdPersonales_homo['ORDEN'].astype(np.str_)
GdPersonales_homo_1 = GdPersonales_homo[GdPersonales_homo['NC4_CC_P3'] == 1]#filtra para que la fomra de adquisición sea solo compra

print(GdPersonales_homo.shape)
print(GdPersonales_homo_1.shape)

#print(GdPersonales_homo_1.shape)
#GdPersonales_homo_1[GdPersonales_homo_1["Id_Person"] == "118512_2"]

# Mostrar el resultado
#compras_gdpersonales_homo_filtrado.head(3)

gdpersonales_homo_objetivo = pd.merge(Id_Person_objetivo,GdPersonales_homo_1, how="inner", left_on="Id_Person", right_on="Id_Person")


# Agrupar después del filtro: Frecuencia de Compra: 1 » 1. Diario2 » 2.1. Varias veces3 » 3. Semanal4 » 4. Quincenal5 » 5. Mensual6 » 6. Bimestral7 » 7. Trimestral9 » 9. Esporádica por semana
gdpersonales_homo_objetivo.groupby("NC4_CC_P6", as_index=False)["DIRECTORIO"].count()

#print(gdpersonales_homo_objetivo.shape)
#gdpersonales_homo_objetivo.head(3)

# Convertir las columnas a string antes de aplicar .str.replace
gdpersonales_homo_objetivo["NC4_CC_P2"] = gdpersonales_homo_objetivo["NC4_CC_P2"].astype(str)
gdpersonales_homo_objetivo["NC4_CC_P5"] = gdpersonales_homo_objetivo["NC4_CC_P5"].astype(str)

# Reemplazar comas por puntos y convertir a números
gdpersonales_homo_objetivo["NC4_CC_P2"] = gdpersonales_homo_objetivo["NC4_CC_P2"].str.replace(",", ".").astype(float).astype("int64")
gdpersonales_homo_objetivo["NC4_CC_P5"] = gdpersonales_homo_objetivo["NC4_CC_P5"].str.replace(",", ".").astype(float).astype("int64")

# Verificar los cambios
#print(gdpersonales_homo_objetivo.dtypes)

# Definir la función con la conversión a int64

def calcular_gdpersonales_mes(row):
    # Convertir las variables NC4_CC_P2 y NC4_CC_P5 a int64
    row["NC4_CC_P2"] = int(row["NC4_CC_P2"])  # Asegurar que sea entero
    row["NC4_CC_P5"] = int(row["NC4_CC_P5"])  # Asegurar que sea entero

    # Aplicar las condiciones según el valor de NH_CGDUCFH_P6
    if row["NC4_CC_P6"] == "1":#diario
        return row["NC4_CC_P5"] * 4.28
    elif row["NC4_CC_P6"] == "2":#VARIAS VECES AL DÍA
        return row["NC4_CC_P5"] * 4.28
    elif row["NC4_CC_P6"] == "3":#Semanal
        return row["NC4_CC_P5"] * 4.28
    elif row["NC4_CC_P6"] == "4":#Quincenal
        return row["NC4_CC_P5"] * 2
    elif row["NC4_CC_P6"] == "5":#Mensual
        return row["NC4_CC_P5"] * 1
    elif row["NC4_CC_P6"] == "6":#Bimestral
        return row["NC4_CC_P5"] / 2
    elif row["NC4_CC_P6"] == "7":#Trimestral
        return row["NC4_CC_P5"] / 3
    elif row["NC4_CC_P6"] == "9":#Esporádica POR SEMANA
        return row["NC4_CC_P5"] * 1
        # Mantener el valor original si ninguna condición aplica


# Aplicar la función fila por fila
#GdUrbanos_2["GASTO_MENSUAL"] = GdUrbanos_2.apply(calcular_gdpersonales_mes, axis=1)

#calcula los gastos presonales mensuales:
gdpersonales_homo_objetivo["gdpersonales_mes"] = gdpersonales_homo_objetivo.apply(calcular_gdpersonales_mes, axis=1)
print(gdpersonales_homo_objetivo.shape)
gdpersonales_homo_objetivo.head(3)

df_resumen = gdpersonales_homo_objetivo[['DIRECTORIO', 'Id_Person', 'DIVISIÓN','gdpersonales_mes']].copy()
print(df_resumen.shape)
df_resumen.head(3)

gdpersonales_homo_objetivo_def = df_resumen.pivot_table(
    index=['DIRECTORIO', 'Id_Person'],
    columns='DIVISIÓN',
    values='gdpersonales_mes',
    aggfunc='sum'
).reset_index()
print(gdpersonales_homo_objetivo_def.shape)
gdpersonales_homo_objetivo_def.head(5)

gdpersonales_homo_objetivo_def.columns = gdpersonales_homo_objetivo_def.columns.astype(str)
gdpersonales_homo_objetivo_def.rename(columns={
    "1": "01_gdperson", "2": "02_gdperson", "3": "03_gdperson",
    "7": "07_gdperson", "8": "08_gdperson", "9": "09_gdperson",
    "12": "12_gdperson"
}, inplace=True)
print(gdpersonales_homo_objetivo_def.shape)
gdpersonales_homo_objetivo_def.head(5)

gdpersonales_homo_objetivo_def.groupby(['Id_Person']).size().reset_index(name='conteo').sort_values(by='conteo', ascending=True).tail(10)#valida si hay duplicados por Id_Person

gdpersonales_homo_objetivo_def.shape

gdpersonales_homo_objetivo_def['directorio'] = (
    gdpersonales_homo_objetivo_def['Id_Person']
    .astype(str)
    .str.split('_')
    .str[0]
)
print(gdpersonales_homo_objetivo_def.shape)
gdpersonales_homo_objetivo_def.head(3)


(524948, 19)
(495839, 19)


(190327, 20)
(190327, 4)
(33858, 9)
(33858, 9)
(33858, 10)


DIVISIÓN,DIRECTORIO,Id_Person,01_gdperson,02_gdperson,03_gdperson,07_gdperson,08_gdperson,09_gdperson,12_gdperson,directorio
0,118277,118277_1,2000.0,NaN,NaN,314384.0,NaN,NaN,NaN,118277
1,118282,118282_1,NaN,NaN,NaN,162640.0,NaN,NaN,NaN,118282
2,118285,118285_2,7276.0,NaN,NaN,4280.0,NaN,NaN,NaN,118285


### DF gastos diarios personales: gdpersonales_homo_objetivo_def


In [22]:
#renombra las columnas directorio para no generar conflicto al hacer el merge coin todas las tablas de gastos
gdpersonales_homo_objetivo_def = gdpersonales_homo_objetivo_def.rename(
    columns={
        "DIRECTORIO": "DIRECTORIO_z",
        "directorio": "directorio_z"
    }
)

# Cantidad de Id_Person únicos
id_person_unicos = gdpersonales_homo_objetivo_def['Id_Person'].nunique()

# Cantidad de directorio únicos
directorio_unicos = gdpersonales_homo_objetivo_def['directorio_z'].nunique()

print("Id_Person únicos:", id_person_unicos)
print("Directorio únicos:", directorio_unicos)

#excel_file_path = '/content/drive/My Drive/ArchivosDatos/gdpersonales_homo_objetivo_def.xlsx'
#gdpersonales_homo_objetivo_def.to_excel(excel_file_path, index=False)  # Set index=False to avoid writing row index


Id_Person únicos: 33858
Directorio únicos: 28712


## Gastos comidas fuera del hogar: GdComidasfueraH


In [23]:
print(GdComidasfueraH.shape)

GdComidasfueraH['Id_Person'] = GdComidasfueraH['DIRECTORIO'].astype(np.str_) + "_" + GdComidasfueraH['ORDEN'].astype(np.str_)
# Mostrar el resultado
#compras_gdpersonales_homo_filtrado.head(3)
GdComidasfueraH_objetivo = pd.merge(Id_Person_objetivo,GdComidasfueraH, how="inner", left_on="Id_Person", right_on="Id_Person")
# Agrupar después del filtro: Frecuencia de Compra: 1 » 1. Diario2 » 2.1. Varias veces3 » 3. Semanal4 » 4. Quincenal5 » 5. Mensual6 » 6. Bimestral7 » 7. Trimestral9 » 9. Esporádica por semana

GdComidasfueraH_objetivo_1 = GdComidasfueraH_objetivo[GdComidasfueraH_objetivo['NH_CGDUCFH_P3'] == 1]#filtra para que la fomra de adquisición sea solo compra
print(GdComidasfueraH_objetivo_1.shape)
GdComidasfueraH_objetivo_1.head(3)

# Convertir las columnas a string antes de aplicar .str.replace
GdComidasfueraH_objetivo_1["NH_CGDUCFH_P2"] = GdComidasfueraH_objetivo_1["NH_CGDUCFH_P2"].astype(str)
GdComidasfueraH_objetivo_1["NH_CGDUCFH_P5"] = GdComidasfueraH_objetivo_1["NH_CGDUCFH_P5"].astype(str)

# Reemplazar comas por puntos y convertir a números
GdComidasfueraH_objetivo_1["NH_CGDUCFH_P2"] = GdComidasfueraH_objetivo_1["NH_CGDUCFH_P2"].str.replace(",", ".").astype(float).astype("int64")
GdComidasfueraH_objetivo_1["NH_CGDUCFH_P5"] = GdComidasfueraH_objetivo_1["NH_CGDUCFH_P5"].str.replace(",", ".").astype(float).astype("int64")

# Verificar los cambios
print(GdComidasfueraH_objetivo_1.dtypes)

# Definir la función con la conversión a int64

def calcular_GdComidasfueraH_mes(row):
    # Convertir las variables NC4_CC_P2 y NC4_CC_P5 a int64
    row["NH_CGDUCFH_P2"] = int(row["NH_CGDUCFH_P2"])  # Asegurar que sea entero
    row["NH_CGDUCFH_P5"] = int(row["NH_CGDUCFH_P5"])  # Asegurar que sea entero

    # Aplicar las condiciones según el valor de NH_CGDUCFH_P6
    if row["NH_CGDUCFH_P6"] == "1":
        return row["NH_CGDUCFH_P5"] * 2.14
    elif row["NH_CGDUCFH_P6"] == "2":
        return row["NH_CGDUCFH_P5"]  * 2.14
    elif row["NH_CGDUCFH_P6"] == "3":
        return row["NH_CGDUCFH_P5"]  * 2.14
    elif row["NH_CGDUCFH_P6"] == "4":
        return row["NH_CGDUCFH_P5"]  * 2
    elif row["NH_CGDUCFH_P6"] == "5":
        return row["NH_CGDUCFH_P5"]  * 1
    elif row["NH_CGDUCFH_P6"] == "6":
        return row["NH_CGDUCFH_P5"]  / 2
    elif row["NH_CGDUCFH_P6"] == "7":
        return row["NH_CGDUCFH_P5"]  / 3
    elif row["NH_CGDUCFH_P6"] == "9":
        return row["NH_CGDUCFH_P5"]  * 1 # Mantener el valor original si ninguna condición aplica


# Aplicar la función fila por fila
#GdUrbanos_2["GASTO_MENSUAL"] = GdUrbanos_2.apply(calcular_gdpersonales_mes, axis=1)

#calcula los gastos presonales mensuales:
GdComidasfueraH_objetivo_1["GdComidasfueraH_mes"] = GdComidasfueraH_objetivo_1.apply(calcular_GdComidasfueraH_mes, axis=1)

# Agrupar y calcular la suma por DIRECTORIO
agrup_GdComidasfueraH_mes = GdComidasfueraH_objetivo_1.groupby("Id_Person", as_index=False)["GdComidasfueraH_mes"].sum()

# Ordenar los resultados por la suma en orden descendente
agrup_GdComidasfueraH_mes = agrup_GdComidasfueraH_mes.sort_values(by="GdComidasfueraH_mes", ascending=False)

# Mostrar los resultados: id person con informacion de gastos personales
print(agrup_GdComidasfueraH_mes.shape)
print(GdComidasfueraH_objetivo_1.shape)

GdComidasfueraH_objetivo_1.groupby("NH_CGDUCFH_P7", as_index=False)["Id_Person"].count()
#1 - Del hogar 2 - Personal

GdComidasfueraH_objetivo_1.groupby("NH_CGDUCFH_P3", as_index=False)["Id_Person"].count()
#1 » 1.Compra
#2 » 2.Recibidos como pago por trabajo
#3 » 3. Regalo o donación
#4 » 4.Intercambio o trueque
#5 » 5.Traidos de la finca o producidos por el hogar
#6 » 6.Tomados de un negocio del hogar
#7 » 7.Otra

#tabla de comidas fuera del hogar personales NH_CGDUCFH_P7: gasto 2 es personal, NH_CGDUCFH_P3:confirma que adquision sean compra
GdComidasfueraH_objetivo_perso = GdComidasfueraH_objetivo_1[
    (GdComidasfueraH_objetivo_1["NH_CGDUCFH_P3"] == 1) &
    (GdComidasfueraH_objetivo_1["NH_CGDUCFH_P7"] == 2)
]


# Agrupar y calcular la suma por ID_PERSON
agrup_GdComidasfueraH_objetivo_perso= GdComidasfueraH_objetivo_perso.groupby("Id_Person", as_index=False)["GdComidasfueraH_mes"].sum()

# Ordenar los resultados por la suma en orden descendente
agrup_GdComidasfueraH_objetivo_perso = agrup_GdComidasfueraH_objetivo_perso.sort_values(by="GdComidasfueraH_mes", ascending=False)
agrup_GdComidasfueraH_objetivo_perso.rename(columns={"GdComidasfueraH_mes": "GdComidasfueraH_mes_perso"}, inplace=True)

# Mostrar los resultados: id person con informacion de gastos personales
agrup_GdComidasfueraH_objetivo_perso.shape

print(agrup_GdComidasfueraH_objetivo_perso.shape)
agrup_GdComidasfueraH_objetivo_perso.head(3)

agrup_GdComidasfueraH_objetivo_perso['directorio'] = (
    agrup_GdComidasfueraH_objetivo_perso['Id_Person']
    .astype(str)
    .str.split('_')
    .str[0]
)
print(agrup_GdComidasfueraH_objetivo_perso.shape)
agrup_GdComidasfueraH_objetivo_perso.head(3)

# Cantidad de Id_Person únicos
id_person_unicos = agrup_GdComidasfueraH_objetivo_perso['Id_Person'].nunique()

# Cantidad de directorio únicos
directorio_unicos = agrup_GdComidasfueraH_objetivo_perso['directorio'].nunique()

print("Id_Person únicos:", id_person_unicos)
print("Directorio únicos:", directorio_unicos)

#tabla de comidas fuera del hogar del hogar NH_CGDUCFH_P7: gasto 1 es hogar, NH_CGDUCFH_P3:confirma que adquision sean compra
GdComidasfueraH_objetivo_hogar = GdComidasfueraH_objetivo_1[
    (GdComidasfueraH_objetivo_1["NH_CGDUCFH_P3"] == 1) &
    (GdComidasfueraH_objetivo_1["NH_CGDUCFH_P7"] == 1)
]


# Agrupar y calcular la suma por ID_PERSON
agrup_GdComidasfueraH_objetivo_hogar= GdComidasfueraH_objetivo_hogar.groupby("Id_Person", as_index=False)["GdComidasfueraH_mes"].sum()

# Ordenar los resultados por la suma en orden descendente
agrup_GdComidasfueraH_objetivo_hogar = agrup_GdComidasfueraH_objetivo_hogar.sort_values(by="GdComidasfueraH_mes", ascending=False)
agrup_GdComidasfueraH_objetivo_hogar.rename(columns={"GdComidasfueraH_mes": "GdComidasfueraH_mes_hogar"}, inplace=True)

# Mostrar los resultados: id person con informacion de gastos personales
print(agrup_GdComidasfueraH_objetivo_hogar.shape)
agrup_GdComidasfueraH_objetivo_hogar.head(4)

agrup_GdComidasfueraH_objetivo_hogar['directorio'] = (
    agrup_GdComidasfueraH_objetivo_hogar['Id_Person']
    .astype(str)
    .str.split('_')
    .str[0]
)
print(agrup_GdComidasfueraH_objetivo_hogar.shape)
agrup_GdComidasfueraH_objetivo_hogar.head(3)


(504031, 15)


(135564, 16)
Id_Person             object
DIRECTORIO             int64
SECUENCIA_ENCUESTA     int64
SECUENCIA_P            int64
ORDEN                  int64
NH_CGDUCFH_P1         object
NH_CGDUCFH_P1_1        int64
NH_CGDUCFH_P1_2        int64
NH_CGDUCFH_P2          int64
NH_CGDUCFH_P3          int64
NH_CGDUCFH_P4         object
NH_CGDUCFH_P5          int64
NH_CGDUCFH_P6         object
NH_CGDUCFH_P7          int64
NH_CGDUCFH_P8          int64
FEX_C                 object
dtype: object


(32391, 2)
(135564, 17)
(10107, 2)
(10107, 3)
Id_Person únicos: 10107
Directorio únicos: 8837
(28949, 2)
(28949, 3)


,Id_Person,GdComidasfueraH_mes_hogar,directorio
27545,693315_1,3321280.0,693315
12640,397984_1,3040940.0,397984
27319,689409_1,2137646.0,689409


### DF gastos diarios del hogar Urbano - Comidas preparadas fuera del hogar (GdComidasfueraH): agrup_GdComidasfueraH_objetivo_hogar


In [24]:
agrup_GdComidasfueraH_objetivo_hogar.shape
#excel_file_path = '/content/drive/My Drive/ArchivosDatos/agrup_GdComidasfueraH_objetivo_hogar.xlsx'
#agrup_GdComidasfueraH_objetivo_hogar.to_excel(excel_file_path, index=False)  # Set index=False to avoid writing row index


(28949, 3)

### DF gastos diarios del hogar Urbano - Comidas preparadas fuera del hogar (GdComidasfueraH): agrup_GdComidasfueraH_objetivo_perso


In [25]:
agrup_GdComidasfueraH_objetivo_perso.shape
#excel_file_path = '/content/drive/My Drive/ArchivosDatos/agrup_GdComidasfueraH_objetivo_perso.xlsx'
#agrup_GdComidasfueraH_objetivo_perso.to_excel(excel_file_path, index=False)  # Set index=False to avoid writing row index


(10107, 3)

## Gastos personales Urbano Comidas preparadas fuera del hogar: GPersonComidasfueraH


In [26]:
#informacion GPersonComidasfueraH
print(GPersonComidasfueraH.shape)
GPersonComidasfueraH['Id_Person'] = GPersonComidasfueraH['DIRECTORIO'].astype(np.str_) + "_" + GPersonComidasfueraH['ORDEN'].astype(np.str_)
# Mostrar el resultado
#compras_gdpersonales_homo_filtrado.head(3)
GPersonComidasfueraH_objetivo = pd.merge(Id_Person_objetivo,GPersonComidasfueraH, how="inner", left_on="Id_Person", right_on="Id_Person")

GPersonComidasfueraH_objetivo_2 = GPersonComidasfueraH_objetivo[GPersonComidasfueraH_objetivo['NH_CGPUCFH_P3'] == 1]#filtra para que la fomra de adquisición sea solo compra
print(GPersonComidasfueraH_objetivo_2.shape)
GPersonComidasfueraH_objetivo_2.head(3)

# Realizar el merge después de la conversión
GPersonComidasfueraH_objetivo_1 = pd.merge(GPersonComidasfueraH_objetivo_2,homologa_producto,how="left",left_on="NH_CGPUCFH_P1_S1",right_on="llave")
print(GPersonComidasfueraH_objetivo_1.shape)
GPersonComidasfueraH_objetivo_1.head(3)

GPersonComidasfueraH_objetivo_1.groupby("DIVISIÓN", as_index=False)["Id_Person"].count()

# Agrupar después del filtro: Frecuencia de Compra: 1 » 1. Diario2 » 2.1. Varias veces3 » 3. Semanal4 » 4. Quincenal5 » 5. Mensual6 » 6. Bimestral7 » 7. Trimestral9 » 9. Esporádica por semana
agrup_freq_GPersonComidasfueraH_objetivo= GPersonComidasfueraH_objetivo_1.groupby("NH_CGPUCFH_P6", as_index=False)["Id_Person"].count()
agrup_freq_GPersonComidasfueraH_objetivo#FRECUENCIA DE COMPRA

# Convertir las columnas a string antes de aplicar .str.replace
GPersonComidasfueraH_objetivo_1["NH_CGPUCFH_P2"] = GPersonComidasfueraH_objetivo_1["NH_CGPUCFH_P2"].astype(str)
GPersonComidasfueraH_objetivo_1["NH_CGPUCFH_P5"] = GPersonComidasfueraH_objetivo_1["NH_CGPUCFH_P5"].astype(str)

# Reemplazar comas por puntos y convertir a números
GPersonComidasfueraH_objetivo_1["NH_CGPUCFH_P2"] = GPersonComidasfueraH_objetivo_1["NH_CGPUCFH_P2"].str.replace(",", ".").astype(float).astype("int64")
GPersonComidasfueraH_objetivo_1["NH_CGPUCFH_P5"] = GPersonComidasfueraH_objetivo_1["NH_CGPUCFH_P5"].str.replace(",", ".").astype(float).astype("int64")

# Verificar los cambios
print(GPersonComidasfueraH_objetivo_1.dtypes)

# Definir la función con la conversión a int64
def calcular_GPersonComidasfueraH_mes(row):
    # Convertir las variables NC4_CC_P2 y NC4_CC_P5 a int64
    row["NH_CGPUCFH_P2"] = int(row["NH_CGPUCFH_P2"])  # Asegurar que sea entero
    row["NH_CGPUCFH_P5"] = int(row["NH_CGPUCFH_P5"])  # Asegurar que sea entero

    # Aplicar las condiciones según el valor de NH_CGDUCFH_P6
    if row["NH_CGPUCFH_P6"] == "1":
        return row["NH_CGPUCFH_P5"]  * 2.14
    elif row["NH_CGPUCFH_P6"] == "2":
        return row["NH_CGPUCFH_P5"]  * 2.14
    elif row["NH_CGPUCFH_P6"] == "3":
        return row["NH_CGPUCFH_P5"]  * 2.14
    elif row["NH_CGPUCFH_P6"] == "4":
        return row["NH_CGPUCFH_P5"]  * 2
    elif row["NH_CGPUCFH_P6"] == "5":
        return row["NH_CGPUCFH_P5"]  * 1
    elif row["NH_CGPUCFH_P6"] == "6":
        return row["NH_CGPUCFH_P5"]  / 2
    elif row["NH_CGPUCFH_P6"] == "7":
        return row["NH_CGPUCFH_P5"]  / 3
    elif row["NH_CGPUCFH_P6"] == "9":
        return row["NH_CGPUCFH_P5"]  * 1 # Mantener el valor original si ninguna condición aplica


# Aplicar la función fila por fila
#GdUrbanos_2["GASTO_MENSUAL"] = GdUrbanos_2.apply(calcular_gdpersonales_mes, axis=1)

#calcula los gastos presonales mensuales:
GPersonComidasfueraH_objetivo_1["01_GPersonComidasfueraH_mes"] = GPersonComidasfueraH_objetivo_1.apply(calcular_GPersonComidasfueraH_mes, axis=1)

df_resumen_GPersonComidasfueraH = GPersonComidasfueraH_objetivo_1[['DIRECTORIO', 'Id_Person', 'DIVISIÓN','01_GPersonComidasfueraH_mes']].copy()
print(df_resumen_GPersonComidasfueraH.shape)
df_resumen_GPersonComidasfueraH.head(3)

df_resumen_GPersonComidasfueraH_objetivo_def = df_resumen_GPersonComidasfueraH.pivot_table(
    index=['DIRECTORIO', 'Id_Person'],
    columns='DIVISIÓN',
    values='01_GPersonComidasfueraH_mes',
    aggfunc='sum'
).reset_index()
print(df_resumen_GPersonComidasfueraH_objetivo_def.shape)
df_resumen_GPersonComidasfueraH_objetivo_def.head(5)

df_resumen_GPersonComidasfueraH_objetivo_def.columns = df_resumen_GPersonComidasfueraH_objetivo_def.columns.astype(str)
df_resumen_GPersonComidasfueraH_objetivo_def.rename(columns={
    "1": "01_gdpersoncomidas_fuera",
    "11": "11_gdpersoncomidas_fuera"
}, inplace=True)
print(df_resumen_GPersonComidasfueraH_objetivo_def.shape)
df_resumen_GPersonComidasfueraH_objetivo_def.head(5)

#df_resumen_GPersonComidasfueraH_objetivo_def

tabla_agrupada_1 = df_resumen_GPersonComidasfueraH_objetivo_def.groupby("Id_Person", as_index=False).count()

print(tabla_agrupada_1.shape)

tabla_agrupada_1 = df_resumen_GPersonComidasfueraH_objetivo_def.groupby("DIRECTORIO", as_index=False).count()
print(tabla_agrupada_1.shape)

a=df_resumen_GPersonComidasfueraH_objetivo_def[df_resumen_GPersonComidasfueraH_objetivo_def["01_gdpersoncomidas_fuera"] >0]

print(a["01_gdpersoncomidas_fuera"].sum())
print(a["01_gdpersoncomidas_fuera"].count())
print(a["11_gdpersoncomidas_fuera"].sum())
print(a["11_gdpersoncomidas_fuera"].count())


(475963, 14)


(140007, 15)
(140007, 22)
Id_Person               object
DIRECTORIO               int64
SECUENCIA_ENCUESTA       int64
SECUENCIA_P              int64
ORDEN                    int64
NH_CGPUCFH_P1           object
NH_CGPUCFH_P1_S1         int64
NH_CGPUCFH_P1_S2         int64
NH_CGPUCFH_P2            int64
NH_CGPUCFH_P3            int64
NH_CGPUCFH_P4           object
NH_CGPUCFH_P5            int64
NH_CGPUCFH_P6           object
NH_CGPUCFH_P8            int64
FEX_C                   object
llave                    int64
DIVISIÓN                 int64
GRUPO                  float64
CLASE                  float64
SUBCLASE               float64
ARTÍCULO               float64
Nombre del Artículo     object
dtype: object


(140007, 4)
(30540, 4)
(30540, 4)
(30540, 4)
(24670, 4)
3638223.0
893
32450458.0
807


### DF gastos personales Urbano - Comidas preparadas fuera del hogar (GPersonComidasfueraH): df_resumen_GPersonComidasfueraH_objetivo_def


In [27]:
df_resumen_GPersonComidasfueraH_objetivo_def.shape
#excel_file_path = '/content/drive/My Drive/ArchivosDatos/df_resumen_GPersonComidasfueraH_objetivo_def.xlsx'
#df_resumen_GPersonComidasfueraH_objetivo_def.to_excel(excel_file_path, index=False)  # Set index=False to avoid writing row index


(30540, 4)

## Gastos menos frecuentes - Medio de pago GmfMP


In [28]:
print(GmfMP.shape)
GmfMP.head(3)

GmfMP['DIRECTORIO'].nunique()

GmfMP["DIRECTORIO"] = GmfMP["DIRECTORIO"].astype(str)
DIRECTORIO_objetivo["DIRECTORIO"] = DIRECTORIO_objetivo["DIRECTORIO"].astype(str)

GmfMP_objetivo = pd.merge(
    GmfMP,
    DIRECTORIO_objetivo,
    how="inner",
    on="DIRECTORIO"
)

print(GmfMP_objetivo.shape)

GmfMP_objetivo.shape

# Cantidad de directorio únicos
directorio_unicos = GmfMP_objetivo['DIRECTORIO'].nunique()

print("Directorio únicos:", directorio_unicos)

GmfMP_objetivo_variables = ["DIRECTORIO","Aportantes_Hogar","P10272S1A1","P10272S1A2","P10272S2A1","P10272S2A2","P10272S3A1","P10272S3A2","P10272S4A1","P10272S4A2","P10272S5A1","P10272S5A2","P10272S6A1","P10272S6A2","P10272S7A1","P10272S7A2","P10272S8A1","P10272S8A2","P10272S9A1","P10272S9A2"]  # Reemplaza con los nombres de las columnas deseadas
# Seleccionar los campos de ViviendaHog antes de realizar el merge
GmfMP_objetivo_def = GmfMP_objetivo[list(map(str, GmfMP_objetivo_variables))].copy()
print(GmfMP_objetivo_def.shape)
GmfMP_objetivo_def.head(3)

GmfMP_objetivo_def.rename(columns={"P10272S1A1":"valor_acueducto","P10272S1A2":"meses_acueducto","P10272S2A1":"valor_basuras","P10272S2A2":"meses_basuras","P10272S3A1":"valor_alcantarillado","P10272S3A2":"meses_alcantarillado"
,"P10272S4A1":"valor_luz","P10272S4A2":"meses_luz","P10272S5A1":"valor_alumbrado_publico","P10272S5A2":"meses_alumbrado_publico","P10272S6A1":"valor_gas","P10272S6A2":"meses_gas","P10272S7A1":"valor_telefono",
"P10272S7A2":"meses_telefono","P10272S8A1":"valor_internet","P10272S8A2":"meses_internet","P10272S9A1":"valor_tv","P10272S9A2":"meses_tv",}, inplace=True)

GmfMP_objetivo_def.head(3)
print(GmfMP_objetivo_def.shape)
GmfMP_objetivo_def.head(3)

# Definir las columnas numéricas
GmfMP_objetivo_variables = [
 'valor_acueducto',
 'meses_acueducto',
 'valor_basuras',
 'meses_basuras',
 'valor_alcantarillado',
 'meses_alcantarillado',
 'valor_luz',
 'meses_luz',
 'valor_alumbrado_publico',
 'meses_alumbrado_publico',
 'valor_gas',
 'meses_gas',
 'valor_telefono',
 'meses_telefono',
 'valor_internet',
 'meses_internet',
 'valor_tv','meses_tv'
]

# Aplicar conversión a numérico con manejo de NaN
GmfMP_objetivo_def[GmfMP_objetivo_variables] = GmfMP_objetivo_def[GmfMP_objetivo_variables].apply(pd.to_numeric, errors="coerce")

# Reemplazar valores NaN con 0 antes de convertir a int
GmfMP_objetivo_def[GmfMP_objetivo_variables] = GmfMP_objetivo_def[GmfMP_objetivo_variables].fillna(0).astype(int)


(87201, 134)


(42788, 135)


Directorio únicos: 42788
(42788, 20)
(42788, 20)


#### Tratamiento de datos:

Cuanto pagó la última vez por el servicio? Superior a 100; 98 incluido en otro servicio o en combo con valor reportado; 99 no sabe el monto; 00 hogar recien confirmado o no paga.

A cuantos meses corresponde ese pago? desde 01 hasta 60; 97 para pagos por dias inferiores al mes; 00, 98, 99: no incluir el numero de meses.


In [29]:
# Definir función para aplicar condiciones
def calcular_acueducto(row):
    # Códigos especiales (no respuesta / no aplica)
    if row["valor_acueducto"] in [98, 99]:
        return 0
    # Códigos especiales (no respuesta / no aplica)
    if row["meses_acueducto"] in [97,98, 99]:
        return 0

    # Si valor_acueducto es nulo o 0
    if pd.isnull(row["valor_acueducto"]) or row["valor_acueducto"] == 0:
        return 0

    # Si valor_acueducto >= 100 y meses_acueducto == 0
    elif row["valor_acueducto"] >= 100 and row["meses_acueducto"] == 0:
        return row["valor_acueducto"]

    # Si meses_acueducto es nulo o 0
    elif pd.isnull(row["meses_acueducto"]) or row["meses_acueducto"] == 0:
        return row["valor_acueducto"]

    # Si meses_acueducto es mayor a 6
    #elif row["meses_acueducto"] > 6:
    #    return row["valor_acueducto"] / 6

    # Caso general
    else:
        return row["valor_acueducto"] / row["meses_acueducto"]

# Aplicar la función fila por fila al DataFrame
GmfMP_objetivo_def["resultado_acueducto"] = GmfMP_objetivo_def.apply(calcular_acueducto, axis=1)

# Definir función para aplicar condiciones
def calcular_basuras(row):
    # Códigos especiales (no respuesta / no aplica)
    if row["valor_basuras"] in [98, 99]:
        return 0
    # Códigos especiales (no respuesta / no aplica)
    if row["meses_basuras"] in [97,98, 99]:
        return 0

    # Si valor_basuras es nulo o 0
    if pd.isnull(row["valor_basuras"]) or row["valor_basuras"] == 0:
        return 0

    # Si valor_basuras >= 100 y meses_basuras == 0
    elif row["valor_basuras"] >= 100 and row["meses_basuras"] == 0:
        return row["valor_basuras"]

    # Si meses_basuras es nulo o 0
    elif pd.isnull(row["meses_basuras"]) or row["meses_basuras"] == 0:
        return row["valor_basuras"]

    # Si meses_basuras es mayor a 6
    #elif row["meses_basuras"] > 6:
    #    return row["valor_basuras"] / 6

    # Caso general
    else:
        return row["valor_basuras"] / row["meses_basuras"]

# Aplicar la función fila por fila al DataFrame
GmfMP_objetivo_def["resultado_basuras"] = GmfMP_objetivo_def.apply(calcular_basuras, axis=1)

# Definir función para aplicar condiciones
def calcular_alcantarillado(row):
    # Códigos especiales (no respuesta / no aplica)
    if row["valor_alcantarillado"] in [98, 99]:
        return 0
    # Códigos especiales (no respuesta / no aplica)
    if row["meses_alcantarillado"] in [97,98, 99]:
        return 0

    # Si valor_alcantarillado es nulo o 0
    if pd.isnull(row["valor_alcantarillado"]) or row["valor_alcantarillado"] == 0:
        return 0

    # Si valor_alcantarillado >= 100 y meses_alcantarillado == 0
    elif row["valor_alcantarillado"] >= 100 and row["meses_alcantarillado"] == 0:
        return row["valor_alcantarillado"]

    # Si meses_alcantarillado es nulo o 0
    elif pd.isnull(row["meses_alcantarillado"]) or row["meses_alcantarillado"] == 0:
        return row["valor_alcantarillado"]

    # Si meses_alcantarillado es mayor a 6
    #elif row["meses_alcantarillado"] > 6:
    #    return row["valor_alcantarillado"] / 6

    # Caso general
    else:
        return row["valor_alcantarillado"] / row["meses_alcantarillado"]

# Aplicar la función fila por fila al DataFrame
GmfMP_objetivo_def["resultado_alcantarillado"] = GmfMP_objetivo_def.apply(calcular_alcantarillado, axis=1)

# Definir función para aplicar condiciones
def calcular_luz(row):
    # Códigos especiales (no respuesta / no aplica)
    if row["valor_luz"] in [98, 99]:
        return 0
    # Códigos especiales (no respuesta / no aplica)
    if row["meses_luz"] in [97,98, 99]:
        return 0

    # Si valor_luz es nulo o 0
    if pd.isnull(row["valor_luz"]) or row["valor_luz"] == 0:
        return 0

    # Si valor_luz >= 100 y meses_luz == 0
    elif row["valor_luz"] >= 100 and row["meses_luz"] == 0:
        return row["valor_luz"]

    # Si meses_luz es nulo o 0
    elif pd.isnull(row["meses_luz"]) or row["meses_luz"] == 0:
        return row["valor_luz"]

    # Si meses_luz es mayor a 6
    #elif row["meses_luz"] > 6:
    #    return row["valor_luz"] / 6

    # Caso general
    else:
        return row["valor_luz"] / row["meses_luz"]

# Aplicar la función fila por fila al DataFrame
GmfMP_objetivo_def["resultado_luz"] = GmfMP_objetivo_def.apply(calcular_luz, axis=1)

# Definir función para aplicar condiciones
def calcular_alumbrado_publico(row):
    # Códigos especiales (no respuesta / no aplica)
    if row["valor_alumbrado_publico"] in [98, 99]:
        return 0
    # Códigos especiales (no respuesta / no aplica)
    if row["meses_alumbrado_publico"] in [97,98, 99]:
        return 0

    # Si valor_alumbrado_publico es nulo o 0
    if pd.isnull(row["valor_alumbrado_publico"]) or row["valor_alumbrado_publico"] == 0:
        return 0

    # Si valor_alumbrado_publico >= 100 y meses_alumbrado_publico == 0
    elif row["valor_alumbrado_publico"] >= 100 and row["meses_alumbrado_publico"] == 0:
        return row["valor_alumbrado_publico"]

    # Si meses_alumbrado_publico es nulo o 0
    elif pd.isnull(row["meses_alumbrado_publico"]) or row["meses_alumbrado_publico"] == 0:
        return row["valor_alumbrado_publico"]

    # Si meses_alumbrado_publico es mayor a 6
    #elif row["meses_alumbrado_publico"] > 6:
    #    return row["valor_alumbrado_publico"] / 6

    # Caso general
    else:
        return row["valor_alumbrado_publico"] / row["meses_alumbrado_publico"]

# Aplicar la función fila por fila al DataFrame
GmfMP_objetivo_def["resultado_alumbrado_publico"] = GmfMP_objetivo_def.apply(calcular_alumbrado_publico, axis=1)

# Definir función para aplicar condiciones
def calcular_gas(row):
    # Códigos especiales (no respuesta / no aplica)
    if row["valor_gas"] in [98, 99]:
        return 0
    # Códigos especiales (no respuesta / no aplica)
    if row["meses_gas"] in [97,98, 99]:
        return 0

    # Si valor_gas es nulo o 0
    if pd.isnull(row["valor_gas"]) or row["valor_gas"] == 0:
        return 0

    # Si valor_gas >= 100 y meses_gas == 0
    elif row["valor_gas"] >= 100 and row["meses_gas"] == 0:
        return row["valor_gas"]

    # Si meses_gas es nulo o 0
    elif pd.isnull(row["meses_gas"]) or row["meses_gas"] == 0:
        return row["valor_gas"]

    # Si meses_gas es mayor a 6
    #elif row["meses_gas"] > 6:
    #    return row["valor_gas"] / 6

    # Caso general
    else:
        return row["valor_gas"] / row["meses_gas"]

# Aplicar la función fila por fila al DataFrame
GmfMP_objetivo_def["resultado_gas"] = GmfMP_objetivo_def.apply(calcular_gas, axis=1)

# Definir función para aplicar condiciones
def calcular_telefono(row):
    # Códigos especiales (no respuesta / no aplica)
    if row["valor_telefono"] in [98, 99]:
        return 0
    # Códigos especiales (no respuesta / no aplica)
    if row["meses_telefono"] in [97,98, 99]:
        return 0

    # Si valor_telefono es nulo o 0
    if pd.isnull(row["valor_telefono"]) or row["valor_telefono"] == 0:
        return 0

    # Si valor_telefono >= 100 y meses_telefono == 0
    elif row["valor_telefono"] >= 100 and row["meses_telefono"] == 0:
        return row["valor_telefono"]

    # Si meses_telefono es nulo o 0
    elif pd.isnull(row["meses_telefono"]) or row["meses_telefono"] == 0:
        return row["valor_telefono"]

    # Si meses_telefono es mayor a 6
    #elif row["meses_telefono"] > 6:
    #    return row["valor_telefono"] / 6

    # Caso general
    else:
        return row["valor_telefono"] / row["meses_telefono"]

# Aplicar la función fila por fila al DataFrame
GmfMP_objetivo_def["resultado_telefono"] = GmfMP_objetivo_def.apply(calcular_telefono, axis=1)

# Definir función para aplicar condiciones
def calcular_internet(row):
    # Códigos especiales (no respuesta / no aplica)
    if row["valor_internet"] in [98, 99]:
        return 0
    # Códigos especiales (no respuesta / no aplica)
    if row["meses_internet"] in [97,98, 99]:
        return 0

    # Si valor_internet es nulo o 0
    if pd.isnull(row["valor_internet"]) or row["valor_internet"] == 0:
        return 0

    # Si valor_internet >= 100 y meses_internet == 0
    elif row["valor_internet"] >= 100 and row["meses_internet"] == 0:
        return row["valor_internet"]

    # Si meses_internet es nulo o 0
    elif pd.isnull(row["meses_internet"]) or row["meses_internet"] == 0:
        return row["valor_internet"]

    # Si meses_internet es mayor a 6
    #elif row["meses_internet"] > 6:
    #    return row["valor_internet"] / 6

    # Caso general
    else:
        return row["valor_internet"] / row["meses_internet"]

# Aplicar la función fila por fila al DataFrame
GmfMP_objetivo_def["resultado_internet"] = GmfMP_objetivo_def.apply(calcular_internet, axis=1)

# Definir función para aplicar condiciones
def calcular_tv(row):
    # Códigos especiales (no respuesta / no aplica)
    if row["valor_tv"] in [98, 99]:
        return 0
    # Códigos especiales (no respuesta / no aplica)
    if row["meses_tv"] in [97,98, 99]:
        return 0

    # Si valor_tv es nulo o 0
    if pd.isnull(row["valor_tv"]) or row["valor_tv"] == 0:
        return 0

    # Si valor_tv >= 100 y meses_tv == 0
    elif row["valor_tv"] >= 100 and row["meses_tv"] == 0:
        return row["valor_tv"]

    # Si meses_tv es nulo o 0
    elif pd.isnull(row["meses_tv"]) or row["meses_tv"] == 0:
        return row["valor_tv"]

    # Si meses_tv es mayor a 6
    #elif row["meses_tv"] > 6:
    #    return row["valor_tv"] / 6

    # Caso general
    else:
        return row["valor_tv"] / row["meses_tv"]

# Aplicar la función fila por fila al DataFrame
GmfMP_objetivo_def["resultado_tv"] = GmfMP_objetivo_def.apply(calcular_tv, axis=1)

GmfMP_objetivo_def["04_GmfMP"] = (GmfMP_objetivo_def["resultado_acueducto"] + GmfMP_objetivo_def["resultado_basuras"]+GmfMP_objetivo_def["resultado_alcantarillado"] + GmfMP_objetivo_def["resultado_luz"]+GmfMP_objetivo_def["resultado_alumbrado_publico"] + GmfMP_objetivo_def["resultado_gas"])
GmfMP_objetivo_def["08_GmfMP"] = GmfMP_objetivo_def["resultado_telefono"] + GmfMP_objetivo_def["resultado_internet"]+GmfMP_objetivo_def["resultado_tv"]

# 1. Calcular ingreso total del hogar
poblacion_objetivo_final['ingreso_hogar'] = (
    poblacion_objetivo_final
    .groupby('DIRECTORIO')['ingreso_mensual_total']
    .transform('sum')
)

# 2. Calcular participación del ingreso en el hogar
poblacion_objetivo_final['part_ingreso_hogar'] = (
    poblacion_objetivo_final['ingreso_mensual_total'] /
    poblacion_objetivo_final['ingreso_hogar']
)

GmfMP_objetivo_def_4 = GmfMP_objetivo_def.merge(poblacion_objetivo_final, how="inner", on="DIRECTORIO")
GmfMP_objetivo_def_4.shape

GmfMP_objetivo_def_4['04_GmfMP_def'] = (
    GmfMP_objetivo_def_4['04_GmfMP'] * GmfMP_objetivo_def_4['part_ingreso_hogar']
)
GmfMP_objetivo_def_4['08_GmfMP_def'] = (
    GmfMP_objetivo_def_4['08_GmfMP'] * GmfMP_objetivo_def_4['part_ingreso_hogar']
)

GmfMP_objetivo_def_valida=GmfMP_objetivo_def_4.copy()

GmfMP_objetivo_def_valida = GmfMP_objetivo_def_valida[
    (GmfMP_objetivo_def_valida[['04_GmfMP_def', '08_GmfMP_def']].fillna(0).sum(axis=1)) != 0
]

GmfMP_objetivo_def_2 = GmfMP_objetivo_def_4[
    ['DIRECTORIO', 'ORDEN', 'Id_Person', '04_GmfMP_def', '08_GmfMP_def']
].copy()

GmfMP_objetivo_def_2.rename(
    columns={
        '04_GmfMP_def': '04_GmfMP',
        '08_GmfMP_def': '08_GmfMP'
    },
    inplace=True
)

# Agrupar y calcular la suma por ID_PERSON
AGRUP_04_GmfMP = GmfMP_objetivo_def_2.groupby("Id_Person", as_index=False)["04_GmfMP"].sum()
AGRUP_08_GmfMP = GmfMP_objetivo_def_2.groupby("Id_Person", as_index=False)["08_GmfMP"].sum()

print(AGRUP_04_GmfMP.shape)
print(AGRUP_08_GmfMP.shape)

AGRUP_04_GmfMP['directorio_04'] = (
    AGRUP_04_GmfMP['Id_Person']
    .astype(str)
    .str.split('_')
    .str[0]
)
print(AGRUP_04_GmfMP.shape)
AGRUP_04_GmfMP.head(3)

AGRUP_08_GmfMP['directorio_08'] = (
    AGRUP_08_GmfMP['Id_Person']
    .astype(str)
    .str.split('_')
    .str[0]
)
print(AGRUP_08_GmfMP.shape)
AGRUP_08_GmfMP.head(3)

AGRUP_04_GmfMP = AGRUP_04_GmfMP[
    (AGRUP_04_GmfMP[['04_GmfMP']].fillna(0).sum(axis=1)) != 0
]
AGRUP_08_GmfMP = AGRUP_08_GmfMP[
    (AGRUP_08_GmfMP[['08_GmfMP']].fillna(0).sum(axis=1)) != 0
]

AGRUP_04_GmfMP['04_GmfMP'] = AGRUP_04_GmfMP['04_GmfMP'].astype(int)
AGRUP_08_GmfMP['08_GmfMP'] = AGRUP_08_GmfMP['08_GmfMP'].astype(int)


(59779, 2)
(59779, 2)
(59779, 3)
(59779, 3)


Nota metodológica: las exportaciones intermedias a Excel se excluyen de la versión reproducible. Si se requieren anexos para tesis, conviene generarlos al final mediante una carpeta de salidas parametrizada.


### DF gastos menos fecuentes medios de pago


In [30]:
print(AGRUP_04_GmfMP.shape)
print(AGRUP_08_GmfMP.shape)


(57030, 3)
(57007, 3)


---
## 💾 Persistencia — Exportación NB02
Guarda los DataFrames limpios para `03_universo_analitico.ipynb`.

In [31]:

# ============================================================
# EXPORTACIÓN DE DATOS — FIN NB02
# Guarda DataFrames limpios para NB03
# ============================================================
import pandas as pd

_exports_02 = {
    "publico_objetivo_5":                       publico_objetivo_5,
    "df_GdUrbano":                              df_GdUrbano,
    "GmenosFreqUrbano_homo_objetivo_def":        GmenosFreqUrbano_homo_objetivo_def,
    "gdpersonales_homo_objetivo_def":           gdpersonales_homo_objetivo_def,
    "agrup_GdComidasfueraH_objetivo_hogar":     agrup_GdComidasfueraH_objetivo_hogar,
    "agrup_GdComidasfueraH_objetivo_perso":     agrup_GdComidasfueraH_objetivo_perso,
    "df_resumen_GPersonComidasfueraH_objetivo_def": df_resumen_GPersonComidasfueraH_objetivo_def,
    "AGRUP_04_GmfMP":                           AGRUP_04_GmfMP,
    "AGRUP_08_GmfMP":                           AGRUP_08_GmfMP,
    "poblacion_objetivo_final":                 poblacion_objetivo_final,
}

for nombre, df in _exports_02.items():
    ruta = PERSIST_DIR / f"nb02_{nombre}.parquet"
    df_save = df.copy()
    for col in df_save.select_dtypes(include='category').columns:
        df_save[col] = df_save[col].astype(str)
    df_save.to_parquet(ruta, index=False)

print(f"✅ NB02 → {len(_exports_02)} DataFrames guardados en {PERSIST_DIR}")


✅ NB02 → 10 DataFrames guardados en /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/02_intermedios
